# Adversarial Attacks & Defences: CNN (ResNet-20) vs. Vision Transformer (ViT) on CIFAR-10

One-click notebook: **Runtime -> Change runtime type -> T4 GPU**, then **Runtime -> Run all**.

Pipeline: pretrained clean CNN/ViT -> native PyTorch FGSM & PGD -> 10-epoch AMP adversarial fine-tuning
(eps = 8/255) -> accuracy-vs-eps curves, cost of adversarial training, transferability, example grid.
Estimated total time on a T4: roughly 20-45 minutes (not yet measured; see README).

**Optional GitHub push:** add a Colab secret named `GITHUB_TOKEN` (fine-grained token, *Contents: read/write*
on the repo only). With it, checkpoints, CSV/MD tables and PNG plots are pushed after every stage and every 5
training epochs. Without it everything still runs, nothing is pushed.

In [ ]:
import base64, os, subprocess  # base64 = text encoding; os = files and folders; subprocess = run terminal commands

REPO_URL = "https://github.com/Mehtab404/DeepLearningProject.git"  # the GitHub project this notebook uses
MOUNT_DRIVE = True  # Colab only: keep saved models on Google Drive so a disconnect loses nothing
RUN_QUICK_TEST = False  # True = run a tiny test first (a few minutes on a GPU)
AUTO_PUSH = True  # upload results to GitHub; only works when a GITHUB_TOKEN secret is found
DRIVE_DIR = "/content/drive/MyDrive/adv_cnn_vs_vit_ckpts"  # Google Drive folder for the backups

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")  # first try: read the token (GitHub's secret password) from the computer's settings
if not GITHUB_TOKEN:  # not found there
    try:  # second try: Colab's secret storage
        from google.colab import userdata  # Colab's secret storage
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or ""  # read the secret (empty text if missing)
    except Exception:  # we are not on Colab
        pass  # ignore it
if not GITHUB_TOKEN:  # still not found
    try:  # third try: Kaggle's secret storage
        from kaggle_secrets import UserSecretsClient  # Kaggle's secret storage
        GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")  # read the secret
    except Exception:  # we are not on Kaggle
        pass  # ignore it

if GITHUB_TOKEN:  # a token was found
    basic = base64.b64encode(f"x-access-token:{GITHUB_TOKEN}".encode()).decode()  # turn the token into the text format git's login needs
    os.environ.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraHeader",  # give the login to git through temporary settings (never saved in a file or log)
                       "GIT_CONFIG_VALUE_0": f"Authorization: Basic {basic}"})  # the login text itself
    print("GITHUB_TOKEN found: results will be pushed to GitHub.")  # tell the user
else:  # no token
    AUTO_PUSH = False  # switch uploading off
    print("No GITHUB_TOKEN found: running without GitHub pushes.")  # tell the user

BASE = "/content" if os.path.isdir("/content") else ("/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd())  # the working folder: Colab's, else Kaggle's, else the current one
REPO_DIR = os.path.join(BASE, "DeepLearningProject")  # folder where the project will be placed
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):  # the project is not downloaded yet
    r = subprocess.run(["git", "clone", REPO_URL, REPO_DIR], capture_output=True, text=True)  # download (clone) the project from GitHub
    if r.returncode != 0:  # the download failed
        print("Clone failed (private repo without token?). Continuing in a fresh local folder, pushes disabled.")  # tell the user
        os.makedirs(REPO_DIR, exist_ok=True)  # create an empty folder instead
        subprocess.run(["git", "init", "-b", "main"], cwd=REPO_DIR, capture_output=True)  # start a new empty git project there
        AUTO_PUSH = False  # switch uploading off
os.chdir(REPO_DIR)  # move into the project folder
subprocess.run(["git", "config", "user.name", "colab-agent"])  # name shown on the uploads
subprocess.run(["git", "config", "user.email", "colab-agent@users.noreply.github.com"])  # email shown on the uploads
os.makedirs("src", exist_ok=True); os.makedirs("checkpoints", exist_ok=True); os.makedirs("outputs", exist_ok=True)  # create the folders for code, saved models and results
print("Working in", os.getcwd())  # show the working folder

if MOUNT_DRIVE and os.path.isdir("/content"):  # on Colab and Drive is wanted
    try:  # attempt to connect Google Drive
        from google.colab import drive  # Colab's Drive tool
        drive.mount("/content/drive")  # connect Google Drive (asks for permission)
    except Exception as e:  # it did not work
        print("Drive mount skipped:", type(e).__name__)  # tell the user
        DRIVE_DIR = ""  # do not use Drive
else:  # not on Colab, or Drive not wanted
    DRIVE_DIR = ""  # do not use Drive

In [ ]:
# next line installs the extra libraries the project needs
!pip -q install timm transformers seaborn tqdm pandas
import torch  # main library for deep learning
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),  # show the library version and whether a GPU is available
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only (use --quick-test)")  # show the GPU name, or a warning that only the CPU is available

In [ ]:
%%writefile src/__init__.py
# WHAT: Marks the src/ folder as a Python package.
# HOW: The presence of __init__.py lets 'from src.attacks import ...' work.
# WHY: All modules import each other with relative imports (from .utils import ...).
# PROS & CONS: + Standard packaging. - The file has no code.
# EXAMPLE: python -m src.models runs the models module as part of the package.
# Adversarial robustness: CNN vs ViT on CIFAR-10  <- this folder is a Python package (a group of code files)

In [ ]:
%%writefile src/utils.py
"""Small helper tools used by the other files."""
# WHAT: Imports libraries for JSON files, paths, random numbers, file copying, terminal commands and arrays.
# HOW: json = text format for logs, subprocess = run git, shutil = copy files, numpy/random/torch = seeds.
# WHY: These small helpers are shared by training, evaluation and the notebook.
# PROS & CONS: + Standard library. - None.
# EXAMPLE: json.dump(obj, f) writes the training history.
import json  # read and write JSON text files
import os  # work with file and folder paths
import random  # Python's random numbers
import shutil  # copy files
import subprocess  # run terminal commands from Python
import sys  # access to the error output

import numpy as np  # fast number lists
import torch  # main library for deep learning

# WHAT: Single project-wide random seed.
# HOW: A constant number used to initialise every random generator.
# WHY: Same seed gives the same evaluation subset and the same random starts (when the call order is the same).
# PROS & CONS: + Repeatable. - A single seed does not measure variation between training runs.
# EXAMPLE: SEED = 42.
SEED = 42  # one fixed starting number for all randomness, so results can be repeated


# WHAT: Fixes all random generators.
# HOW: Seeds Python's random, numpy, PyTorch CPU and all GPUs.
# WHY: Different libraries have separate generators; all must be fixed.
# PROS & CONS: + Repeatable runs. - GPU non-determinism can still cause tiny differences.
# EXAMPLE: set_seed(42).
def set_seed(seed: int = SEED):  # make all random numbers repeatable
    random.seed(seed)  # fix Python's random numbers
    np.random.seed(seed)  # fix NumPy's random numbers
    torch.manual_seed(seed)  # fix PyTorch's random numbers
    if torch.cuda.is_available():  # if a GPU exists
        torch.cuda.manual_seed_all(seed)  # fix the GPU's random numbers too


# WHAT: Chooses where to compute.
# HOW: Returns cuda if a GPU is visible, otherwise cpu.
# WHY: Lets the same code run on Colab (GPU) and a laptop (CPU).
# PROS & CONS: + Portable. - CPU runs are very slow for full training.
# EXAMPLE: device(type='cuda') on a Colab T4.
def get_device() -> torch.device:  # choose where to run the maths
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU (fast) if there is one, else CPU (slow)


# WHAT: Writes a Python dictionary or list to a readable text file.
# HOW: Creates the folder if needed, then json.dump with indent=2.
# WHY: Training history and run summaries must survive Colab disconnects.
# PROS & CONS: + Human-readable. - Not atomic: a crash mid-write could corrupt the file.
# EXAMPLE: save_json({'epoch': 1}, 'checkpoints/training_history.json').
def save_json(obj, path):  # save a Python dictionary or list as a readable text file
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)  # create the folder if it is missing
    with open(path, "w", encoding="utf-8") as f:  # open the file for writing
        json.dump(obj, f, indent=2)  # write the data neatly indented


# WHAT: Reads a JSON file, or returns a default if it does not exist.
# HOW: os.path.exists check, then json.load.
# WHY: The first run has no history file yet and must not crash.
# PROS & CONS: + Safe default. - Silently hides a missing file.
# EXAMPLE: load_json('missing.json', {}) returns {}.
def load_json(path, default=None):  # read a JSON file; if it does not exist, give back `default`
    if not os.path.exists(path):  # file is missing
        return default  # give the fallback value instead of crashing
    with open(path, "r", encoding="utf-8") as f:  # open the file for reading
        return json.load(f)  # read and return its content


# WHAT: Keeps weight files that are too big for GitHub out of git.
# HOW: Scans checkpoints/ for .pth/.pt files above 95 MB and appends them to .git/info/exclude (a private ignore
#      list).
# WHY: GitHub rejects files above 100 MB; one huge file would make every push fail.
# PROS & CONS: + Automatic. - Silent exclusion means large weights must be backed up elsewhere (Drive).
# EXAMPLE: A ViT-Base weight file of about 340 MB is excluded; a 22 MB ViT-Tiny weight file is not.
def _exclude_oversized_files(repo_dir, limit_mb=95):  # tell git to ignore saved-model files that are too big for GitHub (limit about 100 MB)
    # WHAT: Wraps the whole job so it can never crash training.
    # HOW: Any OSError is ignored at the end.
    # WHY: Saving results must never kill a long training run.
    # PROS & CONS: + Robust. - Hides failures.
    # EXAMPLE: A read-only folder just skips the step.
    try:  # never crash because of this
        exclude = os.path.join(repo_dir, ".git", "info", "exclude")  # git's private "ignore" list (not shared)
        if not os.path.isdir(os.path.dirname(exclude)):  # not a git folder
            return  # nothing to do
        existing = open(exclude, encoding="utf-8").read() if os.path.exists(exclude) else ""  # read what is already in the ignore list
        new = []  # big files found
        # WHAT: Looks through every file in checkpoints/.
        # HOW: os.walk lists each folder; the size is compared with the limit in bytes.
        # WHY: Finds the files that are too large.
        # PROS & CONS: + Complete scan. - Walks the folder every push (cheap).
        # EXAMPLE: 95 * 1024 * 1024 bytes = 99,614,720.
        for root, _dirs, files in os.walk(os.path.join(repo_dir, "checkpoints")):  # look through the saved-models folder
            for f in files:  # check every file
                p = os.path.join(root, f)  # full path of the file
                if f.endswith((".pth", ".pt")) and os.path.getsize(p) > limit_mb * 1024 * 1024:  # a saved-model file bigger than the limit
                    rel = os.path.relpath(p, repo_dir).replace(os.sep, "/")  # its path inside the project
                    if rel not in existing:  # not in the ignore list yet
                        new.append(rel)  # remember it
        # WHAT: Writes the new exclusions.
        # HOW: Appends the paths to .git/info/exclude and prints them.
        # WHY: Tells the user which files will not be pushed.
        # PROS & CONS: + Visible. - None.
        # EXAMPLE: [AUTO-PUSH] Excluded from git (>95MB): checkpoints/big.pth
        if new:  # at least one big file found
            with open(exclude, "a", encoding="utf-8") as fh:  # open the ignore list to add lines
                fh.write("\n" + "\n".join(new) + "\n")  # add the big files
            print(f"[AUTO-PUSH] Excluded from git (>{limit_mb}MB): {', '.join(new)}")  # tell the user
    except OSError:  # a file problem happened
        pass  # ignore it quietly


# WHAT: Commits and pushes all changes to GitHub, never raising an error.
# HOW: Runs git add, commit and push through subprocess; every failure is printed and returns False.
# WHY: Colab can disconnect at any time; pushing after each stage protects the results.
# PROS & CONS: + Automatic backup. - git add . stages everything in the folder (including unwanted files if
#              .gitignore is incomplete).
# EXAMPLE: git_auto_push('results: evaluation tables and plots').
def git_auto_push(message: str, repo_dir: str = ".") -> bool:  # saves the work to GitHub automatically; never crashes the training
    # WHAT: Small helper that runs one terminal command.
    # HOW: subprocess.run with captured output so nothing is printed (tokens could appear in output).
    # WHY: Keeps secrets out of the logs.
    # PROS & CONS: + Safe logs. - Errors are only visible through return codes.
    # EXAMPLE: run('git', 'status').
    def run(*cmd):  # small helper that runs one terminal command
        return subprocess.run(cmd, cwd=repo_dir, capture_output=True, text=True)  # run it and collect its output instead of printing it

    # WHAT: Starts the guarded upload.
    # HOW: Everything below is inside a try so unexpected errors are caught.
    # WHY: A failed push should not stop the experiment.
    # PROS & CONS: + Robust. - May hide bugs.
    # EXAMPLE: A network error prints '[AUTO-PUSH] Push failed'.
    try:  # attempt the upload
        # WHAT: Checks the folder is a git repository.
        # HOW: git rev-parse --is-inside-work-tree returns 0 inside a repo.
        # WHY: Avoids confusing errors outside git.
        # PROS & CONS: + Clear message. - None.
        # EXAMPLE: '[AUTO-PUSH] Not a git repository, skipping.'
        if run("git", "rev-parse", "--is-inside-work-tree").returncode != 0:  # check this folder is a git project
            print("[AUTO-PUSH] Not a git repository, skipping.")  # it is not
            return False  # stop
        # WHAT: Excludes big files before staging.
        # HOW: Calls the helper above.
        # WHY: Prevents a rejected push.
        # PROS & CONS: + Safe. - None.
        # EXAMPLE: Run before git add.
        _exclude_oversized_files(repo_dir)  # skip files that are too big for GitHub
        # WHAT: Stages all changes.
        # HOW: git add . adds every new and modified file.
        # WHY: Simple for a research repository.
        # PROS & CONS: + Convenient. - Can add junk files; .gitignore must be correct.
        # EXAMPLE: A new PNG plot is staged.
        run("git", "add", ".")  # mark all changed files to be saved
        # WHAT: Skips empty commits.
        # HOW: git diff --cached --quiet returns 0 when nothing is staged.
        # WHY: No point committing nothing.
        # PROS & CONS: + Clean history. - None.
        # EXAMPLE: '[AUTO-PUSH] No file changes to commit.'
        if run("git", "diff", "--cached", "--quiet").returncode == 0:  # code 0 means nothing changed
            print("[AUTO-PUSH] No file changes to commit.")  # tell the user
            return True  # nothing to upload
        # WHAT: Creates the commit.
        # HOW: git commit -m message.
        # WHY: Records a snapshot with a descriptive message.
        # PROS & CONS: + Traceable history. - None.
        # EXAMPLE: 'chore(train): CNN-AdvTrained 10-epoch adversarial fine-tuning finished'.
        commit = run("git", "commit", "-m", message)  # save a snapshot with the given message
        if commit.returncode != 0:  # saving the snapshot failed
            print(f"[AUTO-PUSH] Commit failed: {commit.stderr.strip()[:300]}")  # show the reason
            return False  # stop
        # WHAT: Uploads the commit to GitHub.
        # HOW: git push origin HEAD; failure details are hidden because they may contain a token.
        # WHY: Protects the credentials.
        # PROS & CONS: + No secrets in logs. - Less debugging detail.
        # EXAMPLE: '[AUTO-PUSH] Successfully pushed to GitHub: ...'
        push = run("git", "push", "origin", "HEAD")  # upload the snapshot to GitHub
        if push.returncode != 0:  # the upload failed
            print("[AUTO-PUSH] Push failed (check remote / token). Commit kept locally.")  # tell the user (the details are hidden because they may contain secrets)
            return False  # stop
        print(f"[AUTO-PUSH] Successfully pushed to GitHub: {message}")  # tell the user it worked
        return True  # success
    # WHAT: Catch-all for unexpected errors.
    # HOW: Prints only the exception type, then returns False.
    # WHY: Never crash the experiment because of git.
    # PROS & CONS: + Robust. - May hide real bugs.
    # EXAMPLE: '[AUTO-PUSH] Skipped due to error: FileNotFoundError'.
    except Exception as e:  # noqa: BLE001 - anything unexpected
        print(f"[AUTO-PUSH] Skipped due to error: {type(e).__name__}")  # tell the user, but keep going
        return False  # stop


# WHAT: Copies a folder's files to Google Drive.
# HOW: Creates the Drive folder, copies each file with copy2.
# WHY: Colab local disk is wiped when the session ends; Drive survives.
# PROS & CONS: + Resume after disconnect. - Overwrites Drive files each call.
# EXAMPLE: sync_to_drive('checkpoints', '/content/drive/MyDrive/adv_ckpts').
def sync_to_drive(src_dir: str, drive_dir: str):  # copy a folder's files to Google Drive, so nothing is lost if Colab disconnects
    if not drive_dir:  # no Drive folder given
        return  # do nothing
    try:  # attempt the copy
        os.makedirs(drive_dir, exist_ok=True)  # create the Drive folder if it is missing
        for name in os.listdir(src_dir):  # go through everything in the source folder
            s = os.path.join(src_dir, name)  # full path of the item
            if os.path.isfile(s):  # only copy files, not folders
                shutil.copy2(s, os.path.join(drive_dir, name))  # copy the file to Drive
    except Exception as e:  # noqa: BLE001 - copying failed
        print(f"[DRIVE] Sync skipped: {type(e).__name__}: {e}", file=sys.stderr)  # tell the user, but keep going


# WHAT: The reverse: brings missing files back from Drive.
# HOW: Copies only files that do not exist locally.
# WHY: Lets a new Colab session continue where the old one stopped.
# PROS & CONS: + No retraining. - Does not update older local copies.
# EXAMPLE: '[DRIVE] Restored cnn_adv.pth'.
def restore_from_drive(dest_dir: str, drive_dir: str):  # the opposite: bring saved files back from Google Drive
    if not drive_dir or not os.path.isdir(drive_dir):  # no Drive folder to read from
        return  # do nothing
    os.makedirs(dest_dir, exist_ok=True)  # create the local folder if it is missing
    for name in os.listdir(drive_dir):  # go through the files on Drive
        d = os.path.join(dest_dir, name)  # where the file should go here
        s = os.path.join(drive_dir, name)  # where it is on Drive
        if os.path.isfile(s) and not os.path.exists(d):  # it is a file and we do not have it yet
            shutil.copy2(s, d)  # copy it here
            print(f"[DRIVE] Restored {name}")  # tell the user

In [ ]:
%%writefile src/models.py
"""Models: the two image-recognising models we compare (a CNN and a ViT)."""
# WHAT: Imports file/time helpers, PyTorch, its layer module and its functional API.
# HOW: os for checkpoint folders, time.sleep for retries, nn.Module as the base class for the wrapper,
#      F.interpolate for resizing.
# WHY: Everything needed to build, wrap, save and load the CNN and the ViT.
# PROS & CONS: + Standard. - None.
# EXAMPLE: F.interpolate(x, size=(224, 224), mode='bilinear') enlarges a 32x32 image.
import os  # work with file and folder paths
import time  # used to wait between retries

import torch  # main library for deep learning
import torch.nn as nn  # building blocks for models
import torch.nn.functional as F  # ready-made maths functions (we use it to resize images)

# WHAT: Per-channel mean and standard deviation of the CIFAR-10 training images.
# HOW: Tuples of three numbers (red, green, blue).
# WHY: The pretrained ResNet-20 was trained on images normalised with exactly these values.
# PROS & CONS: + Matches the pretrained weights. - Wrong values would silently lower accuracy.
# EXAMPLE: (0.4914, 0.4822, 0.4465) is the mean colour of CIFAR-10.
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)  # average red, green, blue value of the training images
CIFAR_STD = (0.2470, 0.2435, 0.2616)  # how much red, green, blue values usually vary from that average
# WHAT: Mean and standard deviation of ImageNet images.
# HOW: Standard torchvision/timm constants.
# WHY: The ImageNet-pretrained ResNet-18 expects this normalisation.
# PROS & CONS: + Matches pretraining. - Only used by the optional ResNet-18 experiment.
# EXAMPLE: (0.485, 0.456, 0.406).
IMAGENET_MEAN = (0.485, 0.456, 0.406)  # average colour of ImageNet pictures (used by the ImageNet-pretrained ResNet-18)
IMAGENET_STD = (0.229, 0.224, 0.225)  # how much ImageNet colours usually vary from that average

# WHAT: Names of the 10 classes in label order.
# HOW: A list; position = label number.
# WHY: Used to print readable predictions in the example figure.
# PROS & CONS: + Readable output. - Must match the dataset order.
# EXAMPLE: CIFAR10_CLASSES[6] == 'frog'.
CIFAR10_CLASSES = ["airplane", "automobile", "bird", "cat", "deer",  # the 10 picture types; label 0 = airplane, 1 = automobile, ...
                   "dog", "frog", "horse", "ship", "truck"]  # ... label 9 = truck

# WHAT: Names of optional external model sources.
# HOW: A HuggingFace model id and the GitHub repo that holds the pretrained ResNet-20.
# WHY: Single place to change the sources.
# PROS & CONS: + Easy to update. - The GitHub download needs internet.
# EXAMPLE: 'chenyaofo/pytorch-cifar-models:master'.
HF_VIT_ID = "aaraki/vit-base-patch16-224-in21k-finetuned-cifar10"  # name of an optional bigger ViT on the HuggingFace website
HUB_REPO = "chenyaofo/pytorch-cifar-models:master"  # GitHub place where the ready-made CNN is downloaded from


# WHAT: Runs a function and retries it if the network fails.
# HOW: Up to 5 attempts, waiting 5 s, 10 s, 15 s ... between them; the last failure is re-raised.
# WHY: Model downloads from GitHub/HuggingFace sometimes fail on Colab.
# PROS & CONS: + Fewer lost runs. - A permanent error still wastes about 1 minute of waiting.
# EXAMPLE: [retry] torch.hub cifar10_resnet20 failed ...; attempt 1/5, retrying in 5s
def _with_retries(fn, what="download", attempts=5, base_delay=5):  # runs fn(); if the internet fails, waits and tries again
    for i in range(1, attempts + 1):  # try up to 5 times
        try:  # attempt the job
            return fn()  # success: give back the result and stop
        except Exception as e:  # noqa: BLE001 - something went wrong (e.g. network problem)
            if i == attempts:  # this was the last try
                raise  # give up and show the error
            wait = base_delay * i  # wait longer after each failure: 5s, 10s, 15s ...
            print(f"[retry] {what} failed ({type(e).__name__}: {str(e)[:120]}); attempt {i}/{attempts}, "  # tell the user what failed
                  f"retrying in {wait}s")  # and how long we wait
            time.sleep(wait)  # pause before trying again


# WHAT: A cover around any classifier so that images outside the model stay plain pixels in [0, 1].
# HOW: forward() optionally resizes, then subtracts the mean and divides by the std, then calls the real model
#      and returns its 10 scores.
# WHY: Attacks add eps directly to pixels. If normalisation lived outside the model, eps would mean different
#      things for different models. Inside the model, eps/255 is exact for CNN and ViT alike.
# PROS & CONS: + Fair, exact epsilon for every architecture. - Every forward pass repeats the normalisation
#              (tiny cost).
# EXAMPLE: wrapper(torch.rand(4, 3, 32, 32)) returns scores of shape (4, 10).
class NormalizedModelWrapper(nn.Module):  # a cover around a model: it resizes and rescales the image first, so outside the model images stay plain 0-to-1
    # WHAT: Stores the real model and the normalisation constants.
    # HOW: register_buffer saves mean and std inside the module without making them learnable.
    # WHY: Buffers move to the GPU with the model and are saved in checkpoints.
    # PROS & CONS: + Travel with the model. - Included in state_dict (harmless).
    # EXAMPLE: wrapper.mean.shape == (1, 3, 1, 1).
    def __init__(self, model, mean=CIFAR_MEAN, std=CIFAR_STD, img_size=None):  # set up the cover
        super().__init__()  # start the basic PyTorch model setup
        self.model = model  # the real model inside (CNN or ViT)
        self.img_size = img_size  # size to enlarge images to (None = keep the size; 224 for the ViT)
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))  # store the average colour inside the model (fixed numbers, not learned)
        self.register_buffer('std', torch.tensor(std).view(1, 3, 1, 1))  # store the colour variation inside the model (fixed numbers, not learned)

    # WHAT: Defines what happens when an image batch is given to the model.
    # HOW: Resize (ViT only), normalise, run the model, return logits.
    # WHY: This is the single entry point that attacks and training use.
    # PROS & CONS: + Clean interface. - None.
    # EXAMPLE: scores = wrapper(x).
    def forward(self, x):  # runs when we give an image to the model
        # WHAT: Enlarges small images for the ViT.
        # HOW: Bilinear interpolation from 32x32 to 224x224, only if the size differs.
        # WHY: ViT-Tiny/16 expects 224x224 input (14x14 patches of 16x16 pixels).
        # PROS & CONS: + Lets a 32x32 dataset use an ImageNet model; the attack still perturbs 32x32 pixels. -
        #              49x more pixels to process, so the ViT is much slower than the CNN; the upsampling may
        #              also smooth the perturbation.
        # EXAMPLE: (4, 3, 32, 32) becomes (4, 3, 224, 224).
        if self.img_size is not None and x.shape[-1] != self.img_size:  # the ViT needs bigger images than the small 32x32 ones
            x = F.interpolate(x, size=(self.img_size, self.img_size), mode='bilinear', align_corners=False)  # enlarge the image smoothly to 224x224
        # WHAT: Normalises the image.
        # HOW: (x - mean) / std per colour channel.
        # WHY: Networks learn faster and predict better with centred inputs.
        # PROS & CONS: + Matches the pretrained weights. - None.
        # EXAMPLE: A pixel 0.5 with mean 0.5, std 0.5 becomes 0.0 (ViT).
        x_norm = (x - self.mean) / self.std  # subtract the average colour and divide by the variation (models learn better this way)
        # WHAT: Calls the real model.
        # HOW: Forward pass on the normalised image.
        # WHY: Produces the class scores.
        # PROS & CONS: + Direct. - None.
        # EXAMPLE: out has 10 scores per image.
        out = self.model(x_norm)  # run the real model
        # WHAT: Returns only the scores.
        # HOW: HuggingFace models wrap them in an object with a .logits field; plain models return a tensor.
        # WHY: Lets both model families share the same interface.
        # PROS & CONS: + Works for both. - Uses a runtime attribute check.
        # EXAMPLE: out.logits.shape == (batch, 10).
        return out.logits if hasattr(out, 'logits') else out  # some models wrap their 10 scores in an object; take the scores out of it


# WHAT: Loads the ResNet-20 for CIFAR-10.
# HOW: torch.hub downloads the architecture (and optionally pretrained weights) from
#      chenyaofo/pytorch-cifar-models, then wraps it with CIFAR normalisation.
# WHY: ResNet-20 is a standard small CIFAR CNN with 0.27M weights; public weights give a strong baseline
#      immediately.
# PROS & CONS: + No training needed (91.5% clean accuracy). - You did not train it yourself; trained models can
#              be loaded with pretrained=False plus a checkpoint.
# EXAMPLE: load_cnn_clean(device, pretrained=False) builds the architecture with random weights.
def load_cnn_clean(device, pretrained=True, arch="cifar10_resnet20"):  # get the normally trained CNN (ResNet-20)
    # WHAT: Downloads the model definition and weights.
    # HOW: torch.hub.load fetches GitHub code; trust_repo=True skips the interactive question.
    # WHY: Single line to get a known-good model.
    # PROS & CONS: + Convenient. - Depends on a third-party repository staying online.
    # EXAMPLE: ~170 MB download is cached under ~/.cache/torch/hub.
    base = _with_retries(lambda: torch.hub.load(HUB_REPO, arch, pretrained=pretrained, trust_repo=True),  # download the ready-made CNN (retry if the internet fails)
                         what="torch.hub " + arch)  # name used in the retry message
    # WHAT: Wraps the CNN so inputs stay in [0, 1].
    # HOW: img_size=None means no resizing.
    # WHY: CNN keeps native 32x32 resolution.
    # PROS & CONS: + Fast. - None.
    # EXAMPLE: model(x) with x in [0, 1].
    model = NormalizedModelWrapper(base, CIFAR_MEAN, CIFAR_STD, img_size=None)  # put the cover around it; the CNN keeps 32x32 images
    # WHAT: Moves the CNN to the GPU/CPU and sets test mode.
    # HOW: eval() freezes BatchNorm statistics and disables dropout.
    # WHY: Attacks and evaluation must see deterministic predictions.
    # PROS & CONS: + Deterministic. - Must call train() again for training.
    # EXAMPLE: model.training is False.
    return model.to(device).eval()  # move it to the GPU/CPU and switch to test mode


# WHAT: Loads a Vision Transformer and says whether it still needs clean fine-tuning.
# HOW: Builds ViT-Tiny/16 from timm (5.7M weights, ImageNet pretrained, new 10-class head) or optionally
#      ViT-Base from HuggingFace.
# WHY: ViT-Tiny is small enough for a free Colab GPU and comparable in spirit to a small CNN.
# PROS & CONS: + Fast to train compared with ViT-Base. - The new head is random, so it must be fine-tuned on
#              CIFAR-10.
# EXAMPLE: model, needs_ft = load_vit_clean(device)  # needs_ft is True for a downloaded ViT.
def load_vit_clean(device, backbone="timm-tiny", pretrained=True):  # get the ViT; also says whether it still needs normal training
    # WHAT: Normalisation constants for ViTs.
    # HOW: Centre at 0.5 with scale 0.5, mapping [0, 1] to [-1, 1].
    # WHY: ViT-Tiny from timm was trained with this normalisation.
    # PROS & CONS: + Matches pretraining. - Different from the CNN's constants (handled inside each wrapper).
    # EXAMPLE: Pixel 1.0 maps to 1.0, pixel 0.0 to -1.0.
    mean, std = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)  # the ViT expects colours centred at 0.5 with a spread of 0.5
    # WHAT: Option 1: the small ViT.
    # HOW: timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=10).
    # WHY: Small enough for a T4 (74.7 minutes of adversarial training).
    # PROS & CONS: + Fast. - Lower capacity than ViT-Base.
    # EXAMPLE: 5.7M parameters.
    if backbone == "timm-tiny":  # option 1: small ViT (fast)
        import timm  # library that contains many ready-made models
        base = _with_retries(  # download the small ViT (retry if the internet fails)
            lambda: timm.create_model("vit_tiny_patch16_224", pretrained=pretrained, num_classes=10),  # 224x224 input cut into 16x16 pieces; 10 outputs for our 10 classes
            what="timm vit_tiny weights")  # name used in the retry message
        needs_ft = pretrained  # a downloaded ViT has a new empty 10-class end part, so it needs a short normal training
    # WHAT: Option 2: the big ViT-Base already trained on CIFAR-10.
    # HOW: Loaded from HuggingFace; no extra fine-tuning needed.
    # WHY: Offered as a stronger alternative.
    # PROS & CONS: + Higher accuracy. - Much slower; weights exceed GitHub's size limit.
    # EXAMPLE: Not used for the reported results.
    elif backbone == "hf-base":  # option 2: bigger ViT from HuggingFace (slower)
        from transformers import AutoConfig, ViTForImageClassification  # tools to load that bigger ViT
        if pretrained:  # download the already-learned version
            base = _with_retries(lambda: ViTForImageClassification.from_pretrained(HF_VIT_ID),  # download it (retry if the internet fails)
                                 what="HF ViT-Base weights")  # name used in the retry message
        else:  # only build an empty model; saved weights will be loaded into it later
            base = ViTForImageClassification(AutoConfig.from_pretrained(HF_VIT_ID))  # same shape, random numbers
        needs_ft = False  # this one is already trained on CIFAR-10
    else:  # any other name is a mistake
        raise ValueError(f"Unknown ViT backbone: {backbone}")  # show an error
    # WHAT: Wraps the ViT with the 224x224 resize.
    # HOW: img_size=224 triggers bilinear upsampling inside forward().
    # WHY: ViT-Tiny/16 requires 224x224 input.
    # PROS & CONS: + Same [0, 1] interface as the CNN. - Slower.
    # EXAMPLE: (4, 3, 32, 32) input -> (4, 10) scores.
    model = NormalizedModelWrapper(base, mean, std, img_size=224)  # put the cover around it; images are enlarged to 224x224
    # WHAT: Returns the model and the 'needs fine-tuning' flag.
    # HOW: Tuple of (model, bool).
    # WHY: main.py trains the ViT only when the flag is True.
    # PROS & CONS: + Explicit. - Callers must unpack two values.
    # EXAMPLE: vit, needs_ft = load_vit_clean(device).
    return model.to(device).eval(), needs_ft  # move to the GPU/CPU, test mode; also say if it needs normal training


# WHAT: Optional comparison model: an ImageNet-pretrained ResNet-18 with the same 224x224 input as the ViT.
# HOW: timm resnet18 with a new 10-class head, wrapped with ImageNet normalisation and bilinear upsampling to
#      224.
# WHY: A size- and input-matched CNN would separate 'architecture' from 'size/pretraining' in the CNN-vs-ViT
#      comparison.
# PROS & CONS: + Fairer comparison with the ViT. - 11M weights and 224x224 inputs make it much slower than
#              ResNet-20; not used in the submitted report.
# EXAMPLE: scripts/extra_cnn_experiments.py --only resnet18.
def load_resnet18_clean(device, pretrained=True):  # get an ImageNet-pretrained ResNet-18 (about 11M weights) that takes 224x224 pictures, like the ViT
    import timm  # library that contains many ready-made models
    base = _with_retries(  # download the weights (retry if the internet fails)
        lambda: timm.create_model("resnet18", pretrained=pretrained, num_classes=10),  # new empty 10-class end part for CIFAR-10
        what="timm resnet18 weights")  # name used in the retry message
    model = NormalizedModelWrapper(base, IMAGENET_MEAN, IMAGENET_STD, img_size=224)  # same cover as the ViT: enlarge to 224x224, normalise inside the model
    # WHAT: Returns the ResNet-18 and whether it still needs fine-tuning.
    # HOW: A downloaded pretrained model has a new random head, so the flag equals `pretrained`.
    # WHY: Same convention as load_vit_clean.
    # PROS & CONS: + Consistent. - None.
    # EXAMPLE: (model, True).
    return model.to(device).eval(), pretrained  # move to the GPU/CPU, test mode; a downloaded model still needs normal training


# WHAT: Saves a model's learned numbers to a file.
# HOW: torch.save(state_dict) after creating the folder.
# WHY: Training takes up to 75 minutes; checkpoints prevent losing it.
# PROS & CONS: + Small files (22 MB ViT-Tiny, 1 MB ResNet-20). - Saves weights only, not the architecture.
# EXAMPLE: save_checkpoint(model, 'checkpoints/cnn_adv.pth').
def save_checkpoint(model, path):  # save the model's learned numbers (weights) to a file
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)  # create the folder if it is missing
    torch.save(model.state_dict(), path)  # write all learned numbers to the file


# WHAT: Loads saved weights into a model of the same shape.
# HOW: torch.load with map_location, load_state_dict, then eval().
# WHY: Lets any session reuse trained models.
# PROS & CONS: + Works on GPU or CPU. - The architecture must match exactly.
# EXAMPLE: load_checkpoint(load_cnn_clean(device, pretrained=False), path, device).
def load_checkpoint(model, path, device):  # load saved weights into a model of the same shape
    model.load_state_dict(torch.load(path, map_location=device))  # read the file and copy the numbers into the model (works on GPU or CPU)
    return model.to(device).eval()  # move to the GPU/CPU and switch to test mode


# WHAT: Quick check that both models accept a 32x32 batch and output 10 scores.
# HOW: Runs a random batch through the CNN and the ViT and asserts the output shape.
# WHY: Catches wiring mistakes before a long run.
# PROS & CONS: + Cheap. - Does not test accuracy.
# EXAMPLE: '[sanity] CNN: input (4, 3, 32, 32) -> output (4, 10) OK'.
def sanity_check(device=None):  # quick check that both models take a 32x32 image and give 10 scores
    device = device or torch.device("cpu")  # use the CPU if no device was given
    # WHAT: Creates 4 random fake images.
    # HOW: Uniform values in [0, 1).
    # WHY: No dataset needed for this test.
    # PROS & CONS: + Offline. - Random noise only.
    # EXAMPLE: Shape (4, 3, 32, 32).
    x = torch.rand(4, 3, 32, 32, device=device)  # 4 fake colour images of size 32x32
    # WHAT: Loads both models for the test.
    # HOW: Downloads weights if they are not cached.
    # WHY: Both must work.
    # PROS & CONS: + Tests the real loaders. - Needs internet on first use.
    # EXAMPLE: cnn, vit.
    cnn = load_cnn_clean(device)  # load the CNN
    vit, _ = load_vit_clean(device)  # load the ViT (ignore the second answer)
    # WHAT: Disables gradient tracking for the check.
    # HOW: No computation graph is built.
    # WHY: Saves memory; no learning is needed.
    # PROS & CONS: + Faster. - None.
    # EXAMPLE: Memory use is about half.
    with torch.no_grad():  # no learning information needed for a simple test (saves memory)
        for name, m in (("CNN", cnn), ("ViT", vit)):  # test both models
            out = m(x)  # get the scores for the 4 fake images
            # WHAT: The actual check.
            # HOW: Output must have exactly 4 rows and 10 columns.
            # WHY: Wrong shapes would break the loss and the attacks.
            # PROS & CONS: + Clear failure message. - None.
            # EXAMPLE: (4, 10) passes.
            assert out.shape == (4, 10), f"{name} output shape {tuple(out.shape)} != (4, 10)"  # must be 4 images x 10 scores, else stop with an error
            print(f"[sanity] {name}: input {tuple(x.shape)} -> output {tuple(out.shape)} OK")  # tell the user it is fine


# WHAT: Entry point for 'python -m src.models'.
# HOW: Runs the sanity check when the file is executed directly.
# WHY: Quick manual test.
# PROS & CONS: + Standard pattern. - None.
# EXAMPLE: python -m src.models
if __name__ == "__main__":  # true only when this file is run directly
    sanity_check()  # run the quick check

In [ ]:
%%writefile src/attacks.py
"""Attacks: small tricks that change an image a tiny bit so the model gets it wrong."""
# WHAT: Imports PyTorch (tensors, autograd) and its functional API F, which every attack in this file needs.
# HOW: torch holds images as tensors and tracks operations so gradients can be computed; F.cross_entropy turns
#      model scores and true labels into one loss number.
# WHY: A gradient attack needs d(loss)/d(pixel). Only an autograd library can compute that for a deep network.
# PROS & CONS: + Standard, GPU-ready, no extra dependency. - Ties the attacks to PyTorch models (not
#              framework-agnostic).
# EXAMPLE: F.cross_entropy(logits of shape (4, 10), y of shape (4,)) returns one scalar loss.
import torch  # main library for deep learning
import torch.nn.functional as F  # ready-made maths functions (we use it for the loss)


# WHAT: Returns d(loss)/d(input image) for a whole batch: the per-pixel direction that makes the model MORE
#       wrong.
# HOW: Copy x, mark it as differentiable, run the model, compute cross-entropy summed over the batch, then
#      torch.autograd.grad(loss, x) returns a tensor shaped exactly like x. NaN values are replaced by 0.
# WHY: FGSM, PGD and RS-FGSM are all built from this one gradient, so a single tested routine keeps them
#      consistent.
# PROS & CONS: + One shared routine; autograd.grad leaves the weights' .grad buffers untouched. - nan_to_num
#              hides numerical problems instead of fixing them.
# EXAMPLE: x of shape (100, 3, 32, 32) gives a gradient of shape (100, 3, 32, 32); its .sign() contains only -1,
#          0 or +1.
def _input_grad(model, x, y, amp=False):  # finds which way to change each pixel so the model becomes MORE wrong
    # WHAT: Makes a private copy of the image batch that autograd will track.
    # HOW: clone() copies the data, detach() cuts any old graph, requires_grad_(True) tells autograd to record
    #      gradients for these pixels.
    # WHY: The original x must stay unchanged, and we want gradients for the IMAGE, not for the model weights.
    # PROS & CONS: + Cannot corrupt the caller's tensor. - One extra copy of the batch in memory.
    # EXAMPLE: x.requires_grad is False before this line and True after it (on the copy).
    x = x.clone().detach().requires_grad_(True)  # copy the image and tell PyTorch we want the push direction for the IMAGE (not the weights)
    # WHAT: Decides whether to use fast half-precision (fp16) maths for this forward pass.
    # HOW: fp16 is used only when amp=True AND the tensor lives on a CUDA GPU (x.is_cuda).
    # WHY: Training attacks run in fp16 for speed on the T4; evaluation attacks use full fp32 so the reported
    #      accuracy is not affected by fp16 gradient noise.
    # PROS & CONS: + Roughly 2x faster and half the memory in training. - fp16 gradients are slightly noisier,
    #              which is why evaluation passes amp=False.
    # EXAMPLE: Evaluation: _input_grad(model, x, y, amp=False) runs in fp32. Training: amp=True on a GPU.
    use_amp = bool(amp) and x.is_cuda  # faster half-precision maths only if asked AND a GPU is used
    # WHAT: Runs the model forward pass, optionally under automatic mixed precision.
    # HOW: torch.autocast switches eligible operations (convolutions, matmuls) to fp16 inside the block;
    #      enabled=False makes it a no-op.
    # WHY: Keeps one code path for CPU, fp32 GPU and fp16 GPU.
    # PROS & CONS: + No duplicated code. - The autocast context adds a small fixed overhead even when disabled.
    # EXAMPLE: logits = model(x) gives a (batch, 10) score table, one score per CIFAR-10 class.
    with torch.autocast(device_type="cuda" if x.is_cuda else "cpu", enabled=use_amp):  # switch on the faster maths when allowed
        logits = model(x)  # model gives 10 scores per image (one score for each class)
    # WHAT: Measures how wrong the model is on the batch.
    # HOW: Cross-entropy of the float32 scores against the true labels, reduction='sum' adds the per-image
    #      losses instead of averaging them.
    # WHY: With a mean, each image's gradient would be divided by the batch size, so the gradient scale would
    #      change with the batch size. Only the sign matters for the attacks, but a sum keeps magnitudes
    #      consistent.
    # PROS & CONS: + Per-image gradients independent of batch size. - Large raw loss values (not meant to be
    #              logged).
    # EXAMPLE: A batch of 100 images with average loss 2.3 gives loss of about 230.
    loss = F.cross_entropy(logits.float(), y, reduction="sum")  # loss = how wrong the model is; "sum" adds it over all images
    # WHAT: Computes the gradient of the loss with respect to every input pixel.
    # HOW: Backpropagation from the loss to x; returns a tuple, [0] picks the gradient for x.
    # WHY: autograd.grad (instead of loss.backward()) returns the gradient directly and does not write into
    #      model.parameters().grad.
    # PROS & CONS: + No side effects on the model. - Only one gradient is returned per call (fine here, we need
    #              just d/dx).
    # EXAMPLE: A pixel with gradient +0.8 would increase the loss if it became brighter.
    grad = torch.autograd.grad(loss, x)[0]  # for every pixel: does raising it make the loss go up or down?
    # WHAT: Returns the gradient with any NaN/Inf cleaned up and detached from the graph.
    # HOW: nan_to_num replaces NaN by 0 (and Inf by large finite numbers); detach() removes autograd history.
    # WHY: A single NaN would otherwise spread through sign() and ruin the adversarial image; fp16 can produce
    #      them.
    # PROS & CONS: + Robust against rare overflow. - A zero gradient means that pixel simply is not moved in
    #              that step.
    # EXAMPLE: tensor([nan, 0.3]) becomes tensor([0.0, 0.3]).
    return torch.nan_to_num(grad.detach())  # replace any broken (NaN) numbers with 0 so they cannot ruin the image


# WHAT: FGSM: the Fast Gradient Sign Method (Goodfellow et al., 2015), a ONE-step attack.
# HOW: x_adv = clip_[0,1]( x + eps * sign( grad_x L ) ). Every pixel moves exactly +eps or -eps in the direction
#      that raises the loss.
# WHY: Cheapest white-box attack; also the baseline against which the stronger PGD is compared.
# PROS & CONS: + Very fast (one forward and one backward pass). - Weak on adversarially trained models: the loss
#              surface is not linear over a large eps, so one big step can overshoot.
# EXAMPLE: eps = 8/255 = 0.0314: a pixel at 0.50 becomes 0.5314 or 0.4686.
def fgsm_attack(model, x, y, eps, amp=False):  # FGSM attack: ONE quick step that pushes every pixel to make the model wrong
    # WHAT: Shortcut for 'no attack allowed'.
    # HOW: With eps = 0 the L-infinity ball contains only the original image, so return an untouched copy.
    # WHY: Makes the eps = 0 point of the accuracy-vs-epsilon curve equal to ordinary clean accuracy.
    # PROS & CONS: + Exact and free. - None.
    # EXAMPLE: fgsm_attack(model, x, y, 0.0) returns a copy equal to x.
    if eps == 0:  # eps = biggest change allowed per pixel; 0 means "do not attack"
        return x.clone().detach()  # give back an untouched copy of the image
    # WHAT: Gets the direction that increases the loss.
    # HOW: One forward and one backward pass through the model.
    # WHY: FGSM needs only the SIGN of this gradient.
    # PROS & CONS: + Single gradient evaluation. - No iteration, so no refinement.
    # EXAMPLE: grad.shape == x.shape == (100, 3, 32, 32).
    grad = _input_grad(model, x, y, amp)  # get the direction that makes the model more wrong
    # WHAT: Takes the single FGSM step.
    # HOW: Add eps times the sign of the gradient to every pixel.
    # WHY: The sign makes every pixel change by the maximum allowed amount, which is the strongest single step
    #      inside the L-infinity ball.
    # PROS & CONS: + Uses the full budget. - Ignores gradient magnitude, so it can waste budget on pixels that
    #              barely matter.
    # EXAMPLE: grad = [-0.3, 0.0, 2.0] -> sign = [-1, 0, 1] -> change = [-eps, 0, +eps].
    x_adv = x.detach() + eps * grad.sign()  # move every pixel by exactly +eps or -eps (sign keeps only the + or - direction)
    # WHAT: Keeps the adversarial image a valid image.
    # HOW: clamp limits every pixel value to the range [0, 1].
    # WHY: Pixel values outside 0..1 are not real images and would break the claim that eps is measured in true
    #      pixel space.
    # PROS & CONS: + Guarantees valid pixels. - Pixels already at 0 or 1 cannot move one way, so the real change
    #              can be smaller than eps.
    # EXAMPLE: 0.99 + 0.0314 = 1.0214 is clamped to 1.0.
    return torch.clamp(x_adv, 0.0, 1.0).detach()  # keep pixel values between 0 (black) and 1 (white)


# WHAT: PGD: Projected Gradient Descent (Madry et al., 2018), a MULTI-step attack and the standard strong
#       L-infinity attack.
# HOW: Start at (optionally randomised) x, repeat `steps` times: x_adv += alpha * sign(grad); project the total
#      change back into [-eps, +eps]; clamp to [0, 1].
# WHY: Many small steps find much worse inputs than one big FGSM step; this is the main robustness test and the
#      attack used inside adversarial training.
# PROS & CONS: + Much stronger than FGSM; still simple and white-box. - Cost grows with `steps` (each step is a
#              forward plus backward pass); still not a guarantee (AutoAttack is stronger).
# EXAMPLE: PGD-10 at eps = 8/255 on 1000 images runs 10 forward+backward passes per image.
def pgd_attack(model, x, y, eps, alpha=None, steps=10, random_start=True, amp=False):  # PGD attack: many small steps, so much stronger than FGSM
    # WHAT: Same 'no attack' shortcut as in FGSM.
    # HOW: Return an unchanged copy when eps is 0.
    # WHY: Keeps the eps = 0 point equal to clean accuracy.
    # PROS & CONS: + Exact and free. - None.
    # EXAMPLE: pgd_attack(..., eps=0.0) returns x unchanged.
    if eps == 0:  # no change allowed
        return x.clone().detach()  # return the image as it is
    # WHAT: Chooses a default step size.
    # HOW: alpha = max(eps / 4, 1/255).
    # WHY: Steps must be small enough to follow the loss surface but large enough to reach the boundary in
    #      `steps` moves.
    # PROS & CONS: + Sensible default for 10 steps. - A fixed rule; with 50 steps a smaller alpha (eps / 10) is
    #              better, which strong_attack_check.py passes explicitly.
    # EXAMPLE: eps = 8/255 gives alpha = 2/255.
    if alpha is None:  # alpha = size of ONE small step
        alpha = max(eps / 4.0, 1.0 / 255.0)  # default step = a quarter of eps (but never below 1/255)
    # WHAT: Freezes the original image as the reference point.
    # HOW: detach() removes gradient history; x is never modified afterwards.
    # WHY: The projection step measures the total change relative to this original.
    # PROS & CONS: + Prevents accidental in-place edits. - None.
    # EXAMPLE: x_adv may move, x stays fixed.
    x = x.detach()  # the original image; it is never changed
    # WHAT: Chooses the starting point of the search.
    # HOW: With random_start, begin at x + uniform noise in [-eps, +eps], clamped to [0, 1]; otherwise begin
    #      exactly at x.
    # WHY: A random start avoids a deterministic path and finds stronger examples; it is part of Madry et al.'s
    #      PGD.
    # PROS & CONS: + Stronger, and restarts become possible (different seed gives a different path). - Results
    #              vary slightly with the random seed (42 vs 42.6 in the report).
    # EXAMPLE: A pixel at 0.50 with eps = 0.0314 may start anywhere in [0.4686, 0.5314].
    if random_start:  # start from a slightly shaken image instead of the exact original
        x_adv = torch.clamp(x + torch.empty_like(x).uniform_(-eps, eps), 0.0, 1.0)  # add random noise between -eps and +eps, keep pixels in 0..1
    else:  # no random start
        x_adv = x.clone()  # start from an exact copy of the original
    # WHAT: The PGD iteration loop.
    # HOW: Each pass: gradient at the current x_adv, one signed step of size alpha, then the projection below.
    # WHY: More iterations let the attack climb the loss surface further.
    # PROS & CONS: + Adaptive: each step uses the gradient at the new point. - Linear cost in `steps`.
    # EXAMPLE: steps = 10 means 10 gradient evaluations.
    for _ in range(steps):  # repeat the small step several times
        # WHAT: Gradient at the CURRENT adversarial image (not at the original).
        # HOW: One forward + backward pass through the model for the batch.
        # WHY: The loss surface is curved; re-evaluating the gradient after each step is what makes PGD stronger
        #      than FGSM.
        # PROS & CONS: + Follows the curve. - One full model pass per step.
        # EXAMPLE: After step 1, x_adv differs from x, so the gradient also differs.
        grad = _input_grad(model, x_adv, y, amp)  # direction that makes the model more wrong, for the current tricked image
        # WHAT: Takes one small signed step that increases the loss.
        # HOW: Add alpha times the sign of the gradient to every pixel.
        # WHY: The sign makes the step an L-infinity step: each pixel moves by exactly alpha.
        # PROS & CONS: + Uses the budget evenly. - Ignores magnitude.
        # EXAMPLE: alpha = 2/255: a pixel moves by 0.00784.
        x_adv = x_adv + alpha * grad.sign()  # take one small step in that direction
        # WHAT: PROJECTION: forces the image back inside the allowed region.
        # HOW: delta = x_adv - x is the total change so far; clamp it to [-eps, +eps] per pixel, then x_adv =
        #      clamp(x + delta, 0, 1).
        # WHY: After several steps the total change could exceed eps. The attack is only valid if every pixel
        #      stays within eps of the original and within the valid pixel range.
        # PROS & CONS: + Guarantees ||x_adv - x||_inf <= eps and 0 <= x_adv <= 1 (checked by
        #              check_perturbation). - Pixels pushed against the boundary stop moving there.
        # EXAMPLE: A pixel at 0.5 with eps = 8/255 can never leave [0.4686, 0.5314], however many steps are
        #          taken.
        delta = torch.clamp(x_adv - x, -eps, eps)  # total change so far, cut off so no pixel moved more than eps
        x_adv = torch.clamp(x + delta, 0.0, 1.0)  # original + allowed change, pixels kept between 0 and 1
    # WHAT: Hands back the final adversarial images.
    # HOW: detach() removes autograd history so later code does not keep the graph alive.
    # WHY: Saves memory and avoids accidental gradient flow into the attack.
    # PROS & CONS: + Lower memory. - None.
    # EXAMPLE: The returned tensor has shape (batch, 3, 32, 32).
    return x_adv.detach()  # the final tricked image


# WHAT: RS-FGSM: random start plus one large FGSM step (Wong et al., 2020), used only to train defences quickly.
# HOW: x0 = x + U(-eps, eps); x_adv = clip( x + clamp( x0 + alpha * sign(grad at x0) - x, -eps, eps ) ), with
#      alpha = 1.25 * eps by default.
# WHY: About 2.5x cheaper than PGD-5 for training (30.1 vs 74.7 minutes for the ViT).
# PROS & CONS: + Fast. - Can cause 'catastrophic overfitting': robust to FGSM but not to PGD, which is exactly
#              what the report found (79.8% under FGSM, 4.9% under PGD-10).
# EXAMPLE: Report result: ViT trained with RS-FGSM scored 4.9% under PGD-10 at 8/255, PGD-5 training gave 42.2%.
def rs_fgsm_attack(model, x, y, eps, alpha=None, amp=False):  # RS-FGSM: random start + ONE step; fast, used while TRAINING the defence
    # WHAT: Same 'no attack' shortcut.
    # HOW: Return an unchanged copy when eps is 0.
    # WHY: Consistent behaviour across all three attacks.
    # PROS & CONS: + Exact and free. - None.
    # EXAMPLE: rs_fgsm_attack(..., eps=0.0) returns x unchanged.
    if eps == 0:  # no change allowed
        return x.clone().detach()  # return the image as it is
    # WHAT: Default step size for RS-FGSM.
    # HOW: alpha = 1.25 * eps.
    # WHY: Wong et al. recommend a step slightly larger than eps to compensate for the random start.
    # PROS & CONS: + Follows the published recipe. - Larger steps are what makes catastrophic overfitting more
    #              likely.
    # EXAMPLE: eps = 8/255 gives alpha = 10/255.
    if alpha is None:  # alpha = size of the one step
        alpha = 1.25 * eps  # default step is a bit bigger than eps (the paper's advice)
    # WHAT: Freezes the original image as the reference point.
    # HOW: detach() removes gradient history.
    # WHY: The final projection is measured against this original.
    # PROS & CONS: + Safe. - None.
    # EXAMPLE: x stays unchanged.
    x = x.detach()  # the original image; it is never changed
    # WHAT: Random starting point.
    # HOW: Add uniform noise in [-eps, +eps] and clamp to [0, 1].
    # WHY: The gradient is taken at a random nearby point, not exactly at x, so the model cannot exploit one
    #      fixed gradient direction.
    # PROS & CONS: + Cheap diversification. - Randomness makes training slightly noisy.
    # EXAMPLE: x0 differs from x by at most 8/255 per pixel.
    x0 = torch.clamp(x + torch.empty_like(x).uniform_(-eps, eps), 0.0, 1.0)  # random start: add random noise, keep pixels in 0..1
    # WHAT: Gradient at the random start.
    # HOW: One forward + backward pass.
    # WHY: RS-FGSM uses exactly ONE gradient (that is why it is cheap).
    # PROS & CONS: + One pass only. - No refinement.
    # EXAMPLE: grad.shape == (batch, 3, 32, 32).
    grad = _input_grad(model, x0, y, amp)  # direction that makes the model more wrong, from the random start
    # WHAT: The single large step plus projection.
    # HOW: Move from x0 by alpha along sign(grad), subtract x to get the total change, clamp it to [-eps, +eps].
    # WHY: Keeps the result inside the same L-infinity ball as PGD, so the attacks are comparable.
    # PROS & CONS: + Valid perturbation guaranteed. - Clipping can remove part of the big step.
    # EXAMPLE: |delta| <= 8/255 for every pixel.
    delta = torch.clamp(x0 + alpha * grad.sign() - x, -eps, eps)  # one step, then cut the total change to within +-eps
    # WHAT: Final adversarial image for RS-FGSM.
    # HOW: x + delta, clamped to [0, 1], detached.
    # WHY: Valid image, no gradient history.
    # PROS & CONS: + Valid and light. - None.
    # EXAMPLE: Same shape as x.
    return torch.clamp(x + delta, 0.0, 1.0).detach()  # original + allowed change, pixels kept between 0 and 1


# WHAT: Safety check that an attack followed its rules.
# HOW: Compute the largest absolute pixel change (L-infinity norm) and assert it is <= eps + tol; assert all
#      pixels are in [0, 1].
# WHY: A bug here would make every robustness number meaningless (an attack that cheats with a larger eps looks
#      stronger than it is).
# PROS & CONS: + Cheap, catches silent errors. - Only checks the rules, not whether the attack is effective.
# EXAMPLE: check_perturbation(x, x_adv, 8/255) returns about 0.03137 for a full-strength PGD.
def check_perturbation(x, x_adv, eps, tol=1e-6):  # safety check: did the attack stay within the rules?
    # WHAT: Largest per-pixel change in the whole batch.
    # HOW: max over |x_adv - x| of all pixels.
    # WHY: The L-infinity norm is exactly the quantity eps bounds.
    # PROS & CONS: + One number summarises the whole batch. - Does not say which image broke the rule.
    # EXAMPLE: linf = 0.0314 when some pixel moved the full 8/255.
    linf = torch.max(torch.abs(x_adv - x)).item()  # the biggest change made to any single pixel
    # WHAT: Fails loudly if any pixel moved more than eps.
    # HOW: tol = 1e-6 allows float rounding.
    # WHY: Rounding makes 8/255 + tiny error possible.
    # PROS & CONS: + Tolerant to float error. - A tolerance could hide an error smaller than 1e-6 (irrelevant
    #              for images).
    # EXAMPLE: linf = 0.03137256 <= 0.03137255 + 1e-6 passes.
    assert linf <= eps + tol, f"L-inf violated: {linf:.8f} > eps {eps:.8f}"  # stop with an error if some pixel changed more than eps (tol = tiny rounding allowance)
    # WHAT: Fails loudly if any pixel left the valid range.
    # HOW: Check the minimum is >= 0 and the maximum is <= 1.
    # WHY: Out-of-range pixels are not real images.
    # PROS & CONS: + Guarantees valid images. - Asserts can be disabled with python -O.
    # EXAMPLE: x_adv.min() == 0.0 and x_adv.max() == 1.0 pass.
    assert 0.0 <= x_adv.min().item() and x_adv.max().item() <= 1.0, "x_adv left the [0, 1] range"  # stop with an error if any pixel is outside 0..1
    # WHAT: Returns the measured largest change for logging or tests.
    # HOW: Plain float.
    # WHY: Lets callers print or compare it.
    # PROS & CONS: + Useful in tests. - None.
    # EXAMPLE: 0.03137254901960784 for eps = 8/255.
    return linf  # report the biggest change


# WHAT: Built-in unit test for all attacks, run with: python -m src.attacks.
# HOW: Builds a tiny model and random images, then checks every attack at several eps values.
# WHY: Lets you verify the attacks anywhere (even on CPU) without downloading models or data.
# PROS & CONS: + Fast (seconds). - Uses a toy model, so it checks the rules, not the strength.
# EXAMPLE: Prints '[attacks] self-test passed' when everything holds.
def self_test(device=None, n=8):  # quick built-in test with fake images; run it with: python -m src.attacks
    # WHAT: Imports PyTorch layers for the toy model, only inside the test.
    # HOW: Local import keeps normal imports light.
    # WHY: Only the test needs layers.
    # PROS & CONS: + No cost in production. - Unusual import position.
    # EXAMPLE: nn.Linear(8, 10) is a layer with 8 inputs and 10 outputs.
    import torch.nn as nn  # building blocks for a tiny model
    # WHAT: Picks the device for the test.
    # HOW: Use the one given, otherwise the CPU.
    # WHY: The test must run anywhere.
    # PROS & CONS: + Portable. - Slow on large inputs (not an issue here).
    # EXAMPLE: self_test() uses the CPU.
    device = device or torch.device("cpu")  # use the CPU if no device was given
    # WHAT: Makes the random numbers repeatable.
    # HOW: Fixes PyTorch's random generator.
    # WHY: A test that changes between runs is hard to debug.
    # PROS & CONS: + Reproducible. - Tests only one random draw.
    # EXAMPLE: The same fake images appear every run.
    torch.manual_seed(0)  # fixed starting number so the random numbers are the same every time
    # WHAT: A toy classifier: one conv layer, ReLU, average pooling, one linear layer.
    # HOW: Turns a 3x32x32 image into 10 scores; .eval() means test mode.
    # WHY: A tiny model is enough to check attack rules.
    # PROS & CONS: + Fast and dependency-free. - Not representative of ResNet or ViT.
    # EXAMPLE: Output shape (8, 10) for 8 images.
    model = nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),  # tiny test model: look at the image, keep positive signals, average them
                          nn.Flatten(), nn.Linear(8, 10)).to(device).eval()  # then flatten and give 10 class scores; eval = test mode
    # WHAT: Creates fake images and labels.
    # HOW: rand gives values in [0, 1); randint gives labels 0..9.
    # WHY: No dataset is needed to test the attacks.
    # PROS & CONS: + Fully offline. - Random images are not natural images.
    # EXAMPLE: x.shape == (8, 3, 32, 32).
    x = torch.rand(n, 3, 32, 32, device=device)  # n fake colour images of size 32x32
    y = torch.randint(0, 10, (n,), device=device)  # n fake labels between 0 and 9
    # WHAT: Tests the attacks at several budgets.
    # HOW: Includes 0 (no attack) up to 16/255, the same budgets as the report.
    # WHY: Edge cases (0) and large budgets are where bugs hide.
    # PROS & CONS: + Covers the whole range. - Not exhaustive.
    # EXAMPLE: eps = 8/255 is the training budget.
    for eps in (0.0, 1 / 255, 4 / 255, 8 / 255, 16 / 255):  # try several attack strengths
        # WHAT: Lists every attack as a small function to test.
        # HOW: FGSM, PGD-10, PGD without random start (3 steps), RS-FGSM.
        # WHY: All attack variants share the same rules.
        # PROS & CONS: + One loop tests all. - Slight duplication of attack settings.
        # EXAMPLE: ('pgd10', lambda: pgd_attack(model, x, y, eps, steps=10)).
        for name, fn in (("fgsm", lambda: fgsm_attack(model, x, y, eps)),  # test the FGSM attack
                         ("pgd10", lambda: pgd_attack(model, x, y, eps, steps=10)),  # test PGD with 10 steps
                         ("pgd-no-rs", lambda: pgd_attack(model, x, y, eps, steps=3, random_start=False)),  # test PGD with 3 steps and no random start
                         ("rs_fgsm", lambda: rs_fgsm_attack(model, x, y, eps))):  # test RS-FGSM
            # WHAT: Runs one attack and checks it.
            # HOW: Call the attack, then check_perturbation verifies the L-infinity and range rules.
            # WHY: Each attack must obey the same rules.
            # PROS & CONS: + Fails at the first violation. - Stops at the first failure only.
            # EXAMPLE: Raises AssertionError if a pixel moved more than eps.
            x_adv = fn()  # run the attack
            linf = check_perturbation(x, x_adv, eps)  # make sure it followed the rules
            # WHAT: Extra test for the eps = 0 case.
            # HOW: Requires the largest change to be exactly 0.
            # WHY: With eps = 0 the image must be returned unchanged.
            # PROS & CONS: + Strict. - None.
            # EXAMPLE: linf == 0.0.
            if eps == 0:  # with eps = 0 nothing should change
                assert linf == 0.0, f"{name}: eps=0 must return x unchanged"  # stop with an error if the image changed
        # WHAT: Progress message per budget.
        # HOW: Prints eps converted back to /255 units.
        # WHY: Shows which budgets passed.
        # PROS & CONS: + Visible progress. - Console noise.
        # EXAMPLE: [attacks] eps=8/255: fgsm / pgd / rs_fgsm all satisfy L-inf <= eps and [0, 1]
        print(f"[attacks] eps={eps * 255:.0f}/255: fgsm / pgd / rs_fgsm all satisfy L-inf <= eps and [0, 1]")  # tell the user this strength passed
    # WHAT: Checks the attacks did not touch the model's weight gradients.
    # HOW: Every parameter's .grad must still be None.
    # WHY: If attacks left gradients on the weights, adversarial training would silently accumulate wrong
    #      updates.
    # PROS & CONS: + Catches a subtle bug. - Only checks this toy model.
    # EXAMPLE: p.grad is None for all 8*3*3*3+8 + 8*10+10 parameters.
    assert all(p.grad is None for p in model.parameters()), "attacks must not accumulate parameter grads"  # attacks must not leave any training information inside the model
    # WHAT: Final success message.
    # HOW: Printed only if no assertion failed.
    # WHY: Clear signal to the user.
    # PROS & CONS: + Obvious. - None.
    # EXAMPLE: [attacks] self-test passed
    print("[attacks] self-test passed")  # all checks passed


# WHAT: Entry point when the file is run directly.
# HOW: Python sets __name__ to '__main__' only for direct runs.
# WHY: Lets 'python -m src.attacks' run the test while imports stay silent.
# PROS & CONS: + Standard pattern. - None.
# EXAMPLE: python -m src.attacks
if __name__ == "__main__":  # true only when this file is run directly
    self_test()  # run the quick test

In [ ]:
%%writefile src/data.py
"""Data: get the CIFAR-10 pictures (60,000 small pictures of 10 kinds of things) ready."""
# WHAT: Imports the libraries used to find files, copy them, wait, handle arrays, build tensors and load
#       CIFAR-10.
# HOW: os = paths, shutil = file copy, time = sleep between retries, numpy = random sampling, torch.utils.data =
#      batching, torchvision = the CIFAR-10 dataset class and image transforms.
# WHY: Data preparation touches the file system, the network and PyTorch's data tools.
# PROS & CONS: + All standard libraries. - torchvision downloads from the University of Toronto server, which
#              can be slow (22 minutes once in Colab).
# EXAMPLE: datasets.CIFAR10(root='data', train=True, download=True) fetches 50,000 training images.
import os  # work with file and folder paths
import shutil  # copy files
import time  # wait between retries

import numpy as np  # fast number lists
import torch  # main library for deep learning
from torch.utils.data import DataLoader, Subset  # DataLoader hands out images in groups; Subset keeps only some images
from torchvision import datasets, transforms  # ready-made picture datasets and picture changes

# WHAT: Imports the project-wide random seed (42).
# HOW: Relative import from src/utils.py.
# WHY: One shared seed makes the 1000-image evaluation subset and the shuffling repeatable.
# PROS & CONS: + Single source of truth. - Changing the seed changes every result.
# EXAMPLE: SEED == 42.
from .utils import SEED  # the fixed starting number for randomness

# WHAT: File name of the CIFAR-10 archive.
# HOW: A plain string constant.
# WHY: Used both for the local copy and the optional Google Drive cache.
# PROS & CONS: + One place to change. - None.
# EXAMPLE: 'cifar-10-python.tar.gz' (about 170 MB).
CIFAR_TAR = "cifar-10-python.tar.gz"  # name of the downloaded dataset file


# WHAT: Makes sure the dataset is on disk, downloading it only if needed.
# HOW: Restore a cached archive from Drive if present, try the torchvision download up to 3 times with growing
#      waits, then cache a copy to Drive.
# WHY: Colab sessions are temporary; caching saves a long re-download after every disconnect.
# PROS & CONS: + Robust to network failures. - Needs the Drive folder to be mounted for the caching benefit.
# EXAMPLE: prepare_data('data', '/content/drive/MyDrive/adv_ckpts').
def prepare_data(root="data", drive_dir=None, attempts=3):  # make sure the dataset is on the computer (download it if not)
    # WHAT: Path where the archive should live locally.
    # HOW: Join the data folder and the archive name.
    # WHY: Used for the cache copy and the existence test.
    # PROS & CONS: + Portable across Windows/Linux. - None.
    # EXAMPLE: data/cifar-10-python.tar.gz
    dst = os.path.join(root, CIFAR_TAR)  # where the dataset file should be on this computer
    # WHAT: Path of the optional Drive backup of the archive.
    # HOW: None if no Drive folder was given.
    # WHY: Allows the same function to work with and without Drive.
    # PROS & CONS: + Optional feature. - None.
    # EXAMPLE: /content/drive/MyDrive/adv_ckpts/dataset/cifar-10-python.tar.gz
    cached = os.path.join(drive_dir, "dataset", CIFAR_TAR) if drive_dir else None  # backup copy on Google Drive (only if a Drive folder is given)
    # WHAT: Creates the data folder if it is missing.
    # HOW: exist_ok=True makes it harmless if the folder exists.
    # WHY: Copying or downloading into a missing folder would fail.
    # PROS & CONS: + Idempotent. - None.
    # EXAMPLE: A first run creates ./data.
    os.makedirs(root, exist_ok=True)  # create the data folder if it is missing
    # WHAT: Restores the archive from Drive when it is not on the local disk yet.
    # HOW: copy2 copies the file together with its timestamps.
    # WHY: Copying from Drive takes seconds; downloading takes minutes.
    # PROS & CONS: + Faster restart. - Uses Drive space (about 170 MB).
    # EXAMPLE: Prints '[data] restored CIFAR-10 archive from Drive cache'.
    if cached and not os.path.exists(dst) and os.path.exists(cached):  # we have a Drive copy but no local copy
        shutil.copy2(cached, dst)  # copy it here (much faster than downloading)
        print("[data] restored CIFAR-10 archive from Drive cache")  # tell the user
    # WHAT: Download loop with retries.
    # HOW: Each pass tries to fetch the training and test sets; on an error it waits 10*i seconds (10 s, 20 s)
    #      and tries again; the last failure is re-raised.
    # WHY: Colab network errors are usually temporary.
    # PROS & CONS: + Survives temporary outages. - Gives up after 3 tries.
    # EXAMPLE: [retry] CIFAR-10 download failed (URLError); attempt 1/3
    for i in range(1, attempts + 1):  # try the download up to 3 times
        try:  # attempt the download
            datasets.CIFAR10(root=root, train=True, download=True)  # get the 50,000 training pictures (skipped if already there)
            datasets.CIFAR10(root=root, train=False, download=True)  # get the 10,000 test pictures (skipped if already there)
            break  # worked, so stop trying
        except Exception as e:  # noqa: BLE001 - the download failed
            if i == attempts:  # this was the last try
                raise  # give up and show the error
            print(f"[retry] CIFAR-10 download failed ({type(e).__name__}); attempt {i}/{attempts}")  # tell the user it will retry
            time.sleep(10 * i)  # wait longer each time (10s, 20s, ...)
    # WHAT: Saves a copy of the archive to Drive for next time.
    # HOW: Creates the Drive folder and copies the archive once.
    # WHY: Next session can use the fast restore above.
    # PROS & CONS: + One-time cost. - Requires write access to Drive.
    # EXAMPLE: Prints '[data] cached CIFAR-10 archive to Drive'.
    if cached and os.path.exists(dst) and not os.path.exists(cached):  # we have a local copy but no Drive copy yet
        os.makedirs(os.path.dirname(cached), exist_ok=True)  # create the Drive folder
        shutil.copy2(dst, cached)  # save a copy to Drive for next time
        print("[data] cached CIFAR-10 archive to Drive")  # tell the user


# WHAT: Builds the object that hands out shuffled training batches.
# HOW: Applies optional augmentation, wraps the 50,000 images in a DataLoader with batch size 128 and shuffling.
# WHY: Mini-batch training needs a stream of random batches.
# PROS & CONS: + Fast with several worker processes. - drop_last discards up to 127 images per epoch.
# EXAMPLE: get_train_loader(128) yields 390 batches per epoch.
def get_train_loader(batch_size=128, root="data", max_samples=0, augment=True, num_workers=2, seed=SEED):  # gives training pictures in shuffled groups (batches)
    # WHAT: List of image transforms to apply, built up step by step.
    # HOW: Augmentations first, then conversion to a tensor.
    # WHY: The order matters: PIL-image transforms must run before ToTensor.
    # PROS & CONS: + Flexible. - None.
    # EXAMPLE: [RandomCrop, RandomHorizontalFlip, ToTensor]
    tfms = []  # list of changes to apply to each picture
    # WHAT: Optional data augmentation.
    # HOW: RandomCrop(32, padding=4) shifts the picture by up to 4 pixels; RandomHorizontalFlip mirrors it half
    #      of the time.
    # WHY: More variety reduces overfitting. These are the standard CIFAR-10 augmentations.
    # PROS & CONS: + Cheap and effective. - Makes training accuracy lower than test-time accuracy on clean
    #              images.
    # EXAMPLE: A cat photo may be shifted 3 pixels left and flipped.
    if augment:  # augment = make small random changes so the model sees more variety
        tfms += [transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()]  # slightly shift the picture; sometimes mirror it left-right
    # WHAT: Converts a picture to a tensor with pixel values in [0, 1].
    # HOW: PIL image 0..255 integers become floats 0..1; layout becomes (3, 32, 32).
    # WHY: The whole project keeps images in [0, 1] so that eps means the same for every model. Mean/std
    #      normalisation happens INSIDE the model wrapper.
    # PROS & CONS: + eps is exact in pixel space. - Each model must normalise internally.
    # EXAMPLE: A pixel of 128 becomes 0.502.
    tfms.append(transforms.ToTensor())  # turn the picture into numbers between 0 and 1 (no other rescaling here)
    # WHAT: Loads the 50,000 training images with the chosen transforms.
    # HOW: torchvision reads the archive and applies the transforms on the fly.
    # WHY: Standard dataset requested by the mentor.
    # PROS & CONS: + Official dataset class. - Needs the download step above.
    # EXAMPLE: len(ds) == 50000.
    ds = datasets.CIFAR10(root=root, train=True, download=True, transform=transforms.Compose(tfms))  # the training dataset with those changes
    # WHAT: Optionally uses only part of the training set (quick tests).
    # HOW: A seeded random permutation picks max_samples positions; Subset keeps only those.
    # WHY: The CPU smoke test must finish in minutes.
    # PROS & CONS: + Deterministic subset. - Not used in full runs (max_samples = 0).
    # EXAMPLE: --max-train-samples 64 keeps 64 images.
    if max_samples and max_samples < len(ds):  # use only part of the data (for quick tests)
        g = np.random.RandomState(seed)  # random generator with a fixed start so the same pictures are picked every time
        idx = g.permutation(len(ds))[:max_samples].tolist()  # shuffle all positions and keep the first max_samples
        ds = Subset(ds, idx)  # keep only those pictures
    # WHAT: Checks if a GPU exists.
    # HOW: pin_memory speeds up CPU to GPU copies.
    # WHY: Only useful with a GPU.
    # PROS & CONS: + Faster transfer. - Uses extra pinned RAM.
    # EXAMPLE: True on a Colab T4.
    pin = torch.cuda.is_available()  # True if a GPU exists (then copying data to it can be faster)
    # WHAT: Creates the batch loader.
    # HOW: shuffle=True gives a new order each epoch; drop_last=True keeps every batch the same size;
    #      persistent_workers keeps helper processes alive.
    # WHY: Equal batch sizes make cosine schedules and BatchNorm statistics stable.
    # PROS & CONS: + Fast and stable. - Loses a few images per epoch.
    # EXAMPLE: batch_size=128, num_workers=2.
    return DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=num_workers,  # hand out groups of batch_size pictures in a new random order each epoch
                      pin_memory=pin, drop_last=True,  # drop_last = skip the last small group so all groups have equal size
                      persistent_workers=num_workers > 0)  # keep the helper processes alive between epochs (faster)


# WHAT: Picks the same number of test images from every class.
# HOW: For each of the 10 classes choose n/10 images at random (seeded, no repeats) and return the sorted
#      positions.
# WHY: A balanced subset avoids a class imbalance that could bias accuracy; the same subset is used for every
#      model so results are comparable.
# PROS & CONS: + Fair and repeatable. - A subset of 1000 has a standard error of up to 1.6 points.
# EXAMPLE: n = 1000 gives 100 images per class.
def stratified_indices(targets, n, seed=SEED):  # picks the same number of test pictures from every class, so the test is fair
    # WHAT: Gets the class labels and the list of classes.
    # HOW: Converts the label list to a numpy array and finds the unique classes.
    # WHY: Needed to sample per class.
    # PROS & CONS: + Simple. - None.
    # EXAMPLE: classes = [0, 1, ..., 9].
    targets = np.asarray(targets)  # list of labels as a number array
    classes = np.unique(targets)  # the different classes: 0 to 9
    # WHAT: Number of images per class.
    # HOW: Integer division n // number of classes, at least 1.
    # WHY: Keeps the classes balanced.
    # PROS & CONS: + Guarantees balance. - n not divisible by 10 loses a few images.
    # EXAMPLE: 1000 // 10 = 100.
    per_class = max(1, n // len(classes))  # pictures per class, e.g. 1000 // 10 = 100
    # WHAT: Seeded random generator.
    # HOW: RandomState(seed) always produces the same sequence.
    # WHY: Makes the subset identical in every session.
    # PROS & CONS: + Reproducible. - Legacy numpy API (still fine).
    # EXAMPLE: Seed 42 picks the same 1000 images every time.
    rng = np.random.RandomState(seed)  # random generator with a fixed start
    # WHAT: Collects the chosen image positions.
    # HOW: Starts empty and grows per class.
    # WHY: Needed to build the final subset.
    # PROS & CONS: + Simple. - None.
    # EXAMPLE: []
    idx = []  # the chosen picture positions
    # WHAT: Samples within each class.
    # HOW: np.where finds all positions of class c; rng.choice picks per_class without replacement.
    # WHY: Sampling without replacement avoids duplicates.
    # PROS & CONS: + Balanced. - Python loop over only 10 classes is cheap.
    # EXAMPLE: 100 images drawn from the 1000 'cat' test images.
    for c in classes:  # go through each class
        cls_idx = np.where(targets == c)[0]  # positions of all pictures of this class
        idx.extend(rng.choice(cls_idx, size=min(per_class, len(cls_idx)), replace=False).tolist())  # pick per_class of them at random, never the same one twice
    # WHAT: Returns positions in increasing order.
    # HOW: Sorting makes the order deterministic.
    # WHY: Same tensor layout in every run.
    # PROS & CONS: + Deterministic. - None.
    # EXAMPLE: [3, 17, 24, ...]
    return sorted(idx)  # give back the positions in order


# WHAT: Builds the fixed evaluation tensors X (images) and Y (labels).
# HOW: Loads the 10,000 test images with ToTensor only (no augmentation), selects the stratified subset, stacks
#      it into one tensor.
# WHY: Attacks are applied batch-wise on this one fixed block, so every model is tested on identical images.
# PROS & CONS: + Fair comparison, fast repeated evaluation. - The subset (N = 1000) is smaller than the full
#              10,000 test set.
# EXAMPLE: X.shape == (1000, 3, 32, 32), Y.shape == (1000,).
def get_eval_tensors(n=1000, root="data", seed=SEED):  # builds the fixed test set used to compare all models
    # WHAT: Loads the test set without random augmentation.
    # HOW: train=False selects the 10,000 test images.
    # WHY: Test images must not be randomly altered.
    # PROS & CONS: + Clean evaluation. - None.
    # EXAMPLE: len(ds) == 10000.
    ds = datasets.CIFAR10(root=root, train=False, download=True, transform=transforms.ToTensor())  # the test pictures as numbers between 0 and 1 (no random changes)
    # WHAT: Chooses which test images to use.
    # HOW: Calls stratified_indices with the labels.
    # WHY: Gives the balanced subset.
    # PROS & CONS: + Balanced. - None.
    # EXAMPLE: 1000 indices.
    idx = stratified_indices(ds.targets, n, seed)  # choose an equal number of pictures from each class
    # WHAT: Stacks the chosen images into one tensor.
    # HOW: ds[i][0] is the i-th image; stack adds a batch dimension.
    # WHY: One tensor is faster to move to the GPU and to batch.
    # PROS & CONS: + Fast. - 12 MB of RAM for 1000 images (about 123 MB for all 10,000).
    # EXAMPLE: torch.Size([1000, 3, 32, 32]).
    X = torch.stack([ds[i][0] for i in idx])  # stack the chosen pictures into one big block of numbers
    # WHAT: Labels of the chosen images as integers.
    # HOW: dtype long is what cross-entropy expects.
    # WHY: Matches the model output indices.
    # PROS & CONS: + Correct dtype. - None.
    # EXAMPLE: tensor([0, 0, ..., 9]).
    Y = torch.tensor([ds.targets[i] for i in idx], dtype=torch.long)  # their true labels as whole numbers
    # WHAT: Returns images and labels.
    # HOW: Tuple of two tensors.
    # WHY: Used by evaluation and plotting.
    # PROS & CONS: + Simple. - None.
    # EXAMPLE: X, Y = get_eval_tensors(1000).
    return X, Y  # pictures and labels

In [ ]:
%%writefile src/defense.py
"""Defence: train the model on tricked images too, so it learns not to be fooled."""
# WHAT: Imports helpers for files and timing, PyTorch, a progress bar, the training attacks and the project
#       utilities.
# HOW: tqdm shows a progress bar, F.cross_entropy is the loss, pgd_attack and rs_fgsm_attack create the
#      adversarial images.
# WHY: Adversarial training needs an attack inside the training loop.
# PROS & CONS: + Reuses the evaluation attacks. - None.
# EXAMPLE: from .attacks import pgd_attack, rs_fgsm_attack.
import os  # work with file and folder paths
import time  # measure how long things take

import torch  # main library for deep learning
import torch.nn.functional as F  # ready-made maths functions (we use it for the loss)
from tqdm import tqdm  # shows a progress bar while training

from .attacks import pgd_attack, rs_fgsm_attack  # the two attacks we use to make tricked images during training
from .models import save_checkpoint  # function that saves the model's learned numbers to a file
from .utils import git_auto_push, load_json, save_json, sync_to_drive  # helper functions from utils.py


# WHAT: Switches on mixed-precision maths only when a GPU is used.
# HOW: torch.autocast(device_type, enabled=...) with a CUDA check.
# WHY: fp16 is about twice as fast on a T4 and halves memory; on CPU it gives no benefit.
# PROS & CONS: + One helper for all loops. - fp16 can lose small gradients (GradScaler compensates).
# EXAMPLE: _autocast(device, True) on CUDA runs convolutions in fp16.
def _autocast(device, enabled):  # switches on faster half-precision maths (only on a GPU)
    return torch.autocast(device_type=device.type, enabled=bool(enabled) and device.type == "cuda")  # on only if asked AND a GPU is used


# WHAT: Chooses the attack used to create adversarial images during training.
# HOW: 'pgd5': 5 steps of size 2/255 with a random start. 'rs_fgsm': one step of size 1.25*eps with a random
#      start.
# WHY: PGD-5 is the stronger and safer defence; RS-FGSM is faster but risked catastrophic overfitting.
# PROS & CONS: + Two interchangeable options. - PGD-5 costs about 2.5x more time for the ViT.
# EXAMPLE: make_train_attack('pgd5', 8/255).
def make_train_attack(method, eps):  # picks which attack makes the tricked images during training
    if method == "pgd5":  # option 1: PGD with 5 small steps (stronger, slower)
        return lambda m, x, y: pgd_attack(m, x, y, eps=eps, alpha=2 / 255, steps=5, random_start=True, amp=True)  # 5 steps of size 2/255 with a random start
    if method == "rs_fgsm":  # option 2: RS-FGSM, one step (faster, a bit less stable)
        return lambda m, x, y: rs_fgsm_attack(m, x, y, eps=eps, alpha=1.25 * eps, amp=True)  # one step of size 1.25 * eps with a random start
    raise ValueError(f"method must be 'pgd5' or 'rs_fgsm', got {method!r}")  # any other name is a mistake, so show an error


# WHAT: Keeps a log of training progress in a JSON file.
# HOW: Loads the old log, adds settings and the new epoch entry (replacing an entry of the same epoch), saves
#      it.
# WHY: Enables resuming and later plotting of training curves.
# PROS & CONS: + Human-readable. - Rewrites the whole file each epoch (small).
# EXAMPLE: checkpoints/training_history.json with train_loss, train_clean_acc, train_robust_acc.
def update_history(history_path, tag, entry=None, **meta):  # keeps a log file of how training went (loss and accuracy per epoch)
    hist = load_json(history_path, {}) or {}  # read the old log (or start an empty one)
    rec = hist.setdefault(tag, {"epochs": []})  # find this model's part of the log (or create it)
    rec.update(meta)  # store extra settings such as method, eps, lr
    if entry is not None:  # an entry = the result of one finished epoch
        rec["epochs"] = [e for e in rec["epochs"] if e["epoch"] != entry["epoch"]] + [entry]  # replace an old entry of the same epoch, then add the new one
    save_json(hist, history_path)  # write the log back to the file


# WHAT: True if a model has already finished all epochs.
# HOW: Reads the log: 'completed' flag set and at least `epochs` entries.
# WHY: Prevents retraining when a finished checkpoint exists.
# PROS & CONS: + Saves hours. - Trusts the log.
# EXAMPLE: is_training_complete(path, 'vit_adv', 10).
def is_training_complete(history_path, tag, epochs):  # True if this model has already finished all its epochs
    rec = (load_json(history_path, {}) or {}).get(tag, {})  # read this model's part of the log
    return bool(rec.get("completed")) and len(rec.get("epochs", [])) >= epochs  # marked finished AND enough epochs are recorded


# WHAT: Adversarial training: trains an already trained model on clean AND adversarial images.
# HOW: For each batch, craft x_adv with the chosen attack (model in eval mode), then take one optimiser step on
#      0.5*CE(clean) + 0.5*CE(adv). Uses AdamW, cosine learning-rate decay, mixed precision and a resumable
#      checkpoint.
# WHY: It implements Madry et al.'s defence: min over weights of the worst-case loss inside the eps-ball. Mixing
#      clean loss limits the clean-accuracy drop (about 3 points).
# PROS & CONS: + Large robustness gain (PGD-10 at 8/255: 0% to 18.3% CNN, 42.2% ViT). - About 6 forward/backward
#              passes per batch; ViT took 74.7 minutes for 10 epochs; clean accuracy drops.
# EXAMPLE: adversarial_finetune(model, loader, epochs=10, eps=8/255, method='pgd5').
def adversarial_finetune(model, train_loader, epochs=10, eps=8 / 255, method="pgd5", lr=1e-4,  # train an already-trained model more, using tricked images too
                         device=None, tag="cnn_adv", ckpt_dir="checkpoints", drive_dir=None,  # device = GPU/CPU, tag = name, ckpt_dir = folder for saved files
                         auto_push_every=0, push_fn=git_auto_push, max_batches=0):  # optional GitHub upload; max_batches only limits the quick test
    # WHAT: Picks the device.
    # HOW: Uses the model's own device if none is given.
    # WHY: Avoids mismatches between model and data.
    # PROS & CONS: + Safe default. - None.
    # EXAMPLE: device(type='cuda').
    device = device or next(model.parameters()).device  # if no device was given, use the one the model is already on
    # WHAT: Moves the model to the chosen device.
    # HOW: Copies weights to GPU memory.
    # WHY: Training on GPU is far faster.
    # PROS & CONS: + Fast. - GPU memory limit.
    # EXAMPLE: T4 has 15 GB.
    model.to(device)  # move the model to the GPU/CPU
    # WHAT: Selects the attack function.
    # HOW: Returns a function (model, x, y) -> x_adv.
    # WHY: Training code stays independent of the attack type.
    # PROS & CONS: + Flexible. - None.
    # EXAMPLE: PGD-5.
    attack = make_train_attack(method, eps)  # choose the attack used to make tricked images
    # WHAT: Mixed precision flag.
    # HOW: True on CUDA.
    # WHY: Speed and memory.
    # PROS & CONS: + 2x faster. - Needs GradScaler.
    # EXAMPLE: True on Colab.
    use_amp = device.type == "cuda"  # use faster half-precision maths only on a GPU
    # WHAT: File names for the final weights, the resume state and the log.
    # HOW: Built from the tag (cnn_adv or vit_adv).
    # WHY: Separate files let a crashed run resume.
    # PROS & CONS: + Resumable. - Extra disk (optimizer state is 2x model size).
    # EXAMPLE: checkpoints/vit_adv.pth, vit_adv_state.pt.
    ckpt_path = os.path.join(ckpt_dir, f"{tag}.pth")  # file for the final learned numbers (weights)
    state_path = os.path.join(ckpt_dir, f"{tag}_state.pt")  # file with everything needed to continue after a crash
    hist_path = os.path.join(ckpt_dir, "training_history.json")  # the log file of loss and accuracy
    # WHAT: The weight-update rule.
    # HOW: AdamW with learning rate 1e-4 and weight decay 1e-4.
    # WHY: Small learning rate fine-tunes pretrained weights gently; decay stops weights growing.
    # PROS & CONS: + Robust default for CNN and ViT. - Needs tuning for training from scratch.
    # EXAMPLE: lr = 0.0001.
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)  # the tool that updates the weights; lr = how big each update is; weight_decay stops weights growing too large
    # WHAT: Learning-rate plan.
    # HOW: Cosine annealing from the initial value to 0 over the epochs.
    # WHY: Large steps early, small steps late give a smoother end.
    # PROS & CONS: + No manual schedule. - Assumes the planned number of epochs.
    # EXAMPLE: Epoch 1: 9.8e-5, epoch 10: 0.
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)  # slowly lowers the learning rate towards 0 as epochs pass
    # WHAT: Protects small fp16 gradients.
    # HOW: Multiplies the loss by a large factor before backward and divides afterwards.
    # WHY: fp16 cannot represent very small numbers.
    # PROS & CONS: + Stable mixed precision. - Rare skipped steps when overflow occurs.
    # EXAMPLE: scale = 65536 initially.
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)  # stops very small numbers becoming 0 in half-precision maths
    # WHAT: Resume logic.
    # HOW: If a state file exists, restore model, optimiser, scheduler, scaler and the epoch counter.
    # WHY: Colab may disconnect during a 75-minute run.
    # PROS & CONS: + Lose at most one epoch. - State file is large.
    # EXAMPLE: '[vit_adv] Resuming from epoch 4/10'.
    start_epoch = 0  # begin from the first epoch unless we find a saved state
    if os.path.exists(state_path):  # a saved state means an earlier run stopped half-way
        st = torch.load(state_path, map_location=device)  # load that saved state
        model.load_state_dict(st["model"])  # put the saved weights back into the model
        optimizer.load_state_dict(st["optimizer"])  # restore the weight-updating tool
        scheduler.load_state_dict(st["scheduler"])  # restore the learning-rate plan
        scaler.load_state_dict(st["scaler"])  # restore the small-number protector
        start_epoch = st["epoch"]  # continue from the epoch where it stopped
        print(f"[{tag}] Resuming from epoch {start_epoch}/{epochs}")  # tell the user training is continuing
    # WHAT: Writes the run settings into the log and marks it unfinished.
    # HOW: Stores method, eps, learning rate and device.
    # WHY: Documents exactly how a result was produced.
    # PROS & CONS: + Reproducibility. - None.
    # EXAMPLE: method='pgd5', eps=0.0314.
    update_history(hist_path, tag, completed=False, method=method, eps=eps, lr=lr, target_epochs=epochs,  # write the settings into the log and mark "not finished yet"
                   amp=use_amp, device=str(device))  # also store whether half-precision maths and which device are used
    # WHAT: The epoch loop: one pass over all 50,000 training images.
    # HOW: Repeats batch training, logging and checkpointing.
    # WHY: Standard training structure.
    # PROS & CONS: + Resumable via start_epoch. - None.
    # EXAMPLE: 10 epochs.
    for epoch in range(start_epoch, epochs):  # one loop = one epoch = one full pass over all training images
        # WHAT: Starts the stopwatch and the running totals for this epoch.
        # HOW: Totals of loss, clean-correct, adversarial-correct and image count.
        # WHY: Needed for the log and the progress bar.
        # PROS & CONS: + Cheap. - None.
        # EXAMPLE: seconds = 447 for the ViT.
        t0 = time.time()  # note the start time of this epoch
        tot_loss = tot_clean = tot_adv = n = 0  # running totals: loss, correct normal guesses, correct tricked guesses, images seen
        # WHAT: Progress bar.
        # HOW: Wraps the loader.
        # WHY: Shows remaining time during long runs.
        # PROS & CONS: + Visible progress. - Clutters logs.
        # EXAMPLE: [vit_adv] Epoch 3/10 ... loss 0.72
        pbar = tqdm(train_loader, desc=f"[{tag}] Epoch {epoch + 1}/{epochs}", leave=False)  # progress bar over the batches
        # WHAT: Loop over batches.
        # HOW: x = images, y = labels.
        # WHY: One optimiser step per batch.
        # PROS & CONS: + Standard. - None.
        # EXAMPLE: 390 batches of 128.
        for b, (x, y) in enumerate(pbar):  # take one batch (a small group) of images x and their labels y
            # WHAT: Stops early in quick tests.
            # HOW: Only active when max_batches > 0.
            # WHY: Smoke tests finish in minutes.
            # PROS & CONS: + Fast tests. - Must be 0 in real runs.
            # EXAMPLE: max_batches=2 in --quick-test.
            if max_batches and b >= max_batches:  # only the quick test sets a limit
                break  # stop this epoch early
            # WHAT: Moves the batch to the GPU.
            # HOW: non_blocking overlaps copying and computing.
            # WHY: Data starts on the CPU.
            # PROS & CONS: + Faster. - Needs pinned memory.
            # EXAMPLE: x is a tensor in GPU memory after this line.
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)  # move the batch to the GPU/CPU
            # WHAT: Test mode while crafting adversarial images.
            # HOW: BatchNorm uses running statistics, dropout is off.
            # WHY: The attack must target the model as it will be used at test time and must not change
            #      BatchNorm statistics.
            # PROS & CONS: + Deterministic attack. - Slight mismatch with train-mode statistics.
            # EXAMPLE: model.training is False.
            model.eval()  # test mode while making tricked images (so the model does not change)
            # WHAT: Crafts the adversarial batch.
            # HOW: x_adv = attack(model, x, y) runs 5 forward/backward passes (PGD-5) in fp16.
            # WHY: This is the inner maximisation of adversarial training.
            # PROS & CONS: + Cheap in fp16. - Largest cost of training (about 5 of the 6 passes).
            # EXAMPLE: x_adv.shape == x.shape; |x_adv - x| <= 8/255.
            with _autocast(device, use_amp):  # faster maths if a GPU is used
                x_adv = attack(model, x, y)  # make the tricked version of every image in the batch
            # WHAT: Back to training mode.
            # HOW: BatchNorm updates statistics, dropout is on.
            # WHY: Weights are updated now.
            # PROS & CONS: + Standard. - None.
            # EXAMPLE: model.training is True.
            model.train()  # back to learning mode
            # WHAT: Clears old gradients.
            # HOW: Sets them to None (cheaper than zeros).
            # WHY: Gradients would otherwise accumulate across batches.
            # PROS & CONS: + Saves memory. - None.
            # EXAMPLE: None.
            optimizer.zero_grad(set_to_none=True)  # erase the update directions left from the previous batch
            # WHAT: Forward pass on clean and adversarial images together.
            # HOW: torch.cat stacks them into one batch of 256, giving one forward pass.
            # WHY: One pass is faster than two.
            # PROS & CONS: + Faster. - BatchNorm statistics mix clean and adversarial images.
            # EXAMPLE: logits.shape == (256, 10).
            with _autocast(device, use_amp):  # faster maths if a GPU is used
                logits = model(torch.cat([x, x_adv], dim=0))  # give normal and tricked images together in one go; get 10 scores per image
            # WHAT: Loss in full precision.
            # HOW: Converts fp16 scores to fp32.
            # WHY: Cross-entropy is numerically sensitive.
            # PROS & CONS: + Accurate. - Tiny cost.
            # EXAMPLE: float32.
            logits = logits.float()  # use normal-precision numbers for the loss
            # WHAT: Remembers the batch size.
            # HOW: Used to split clean (first half) from adversarial (second half).
            # WHY: Needed for the loss.
            # PROS & CONS: + Simple. - None.
            # EXAMPLE: 128.
            bs = x.size(0)  # bs = number of images in the batch
            # WHAT: THE training objective: 0.5 * CE(clean) + 0.5 * CE(adversarial).
            # HOW: Average of the loss on clean images and on their adversarial versions.
            # WHY: A purely adversarial loss lowers clean accuracy a lot; mixing 50/50 keeps the clean drop near
            #      3 points.
            # PROS & CONS: + Controlled accuracy-robustness trade-off. - Less robust than 100% adversarial loss.
            # EXAMPLE: Epoch 10 CNN: 0.99; ViT: 0.52.
            loss = 0.5 * F.cross_entropy(logits[:bs], y) + 0.5 * F.cross_entropy(logits[bs:], y)  # loss = half "mistakes on normal images" + half "mistakes on tricked images"
            # WHAT: Computes how every weight should change.
            # HOW: Backpropagation through the scaled loss.
            # WHY: Standard learning step.
            # PROS & CONS: + Stable. - None.
            # EXAMPLE: Gradients exist for all weights.
            scaler.scale(loss).backward()  # work out how every weight should change (made bigger first so small numbers survive)
            # WHAT: Applies the weight update.
            # HOW: AdamW update using the unscaled gradients.
            # WHY: Learning.
            # PROS & CONS: + Skips steps with overflow. - Occasionally skips one batch.
            # EXAMPLE: Weights change by about lr.
            scaler.step(optimizer)  # actually change the weights
            # WHAT: Adjusts the loss scale.
            # HOW: Increases the scale slowly, halves it after an overflow.
            # WHY: Keeps fp16 stable.
            # PROS & CONS: + Automatic. - None.
            # EXAMPLE: 65536 to 32768 after overflow.
            scaler.update()  # adjust the protector for the next batch
            # WHAT: Updates the running totals.
            # HOW: Loss times batch size, plus the number of correctly classified clean and adversarial images.
            # WHY: Gives epoch averages for the log.
            # PROS & CONS: + Cheap. - .item() forces a GPU sync.
            # EXAMPLE: train_clean_acc = 100 * tot_clean / n.
            tot_loss += loss.item() * bs  # add this batch's loss to the running total
            tot_clean += (logits[:bs].argmax(1) == y).sum().item()  # count normal images guessed right (highest score = the guess)
            tot_adv += (logits[bs:].argmax(1) == y).sum().item()  # count tricked images guessed right
            n += bs  # count the images seen so far
            # WHAT: Shows live numbers on the progress bar.
            # HOW: Loss, clean accuracy and adversarial accuracy.
            # WHY: Watch training without waiting for the epoch end.
            # PROS & CONS: + Feedback. - None.
            # EXAMPLE: loss=0.720 clean=97.3% robust=47.6%
            pbar.set_postfix(loss=f"{tot_loss / n:.3f}", clean=f"{100 * tot_clean / n:.1f}%",  # show the loss and the normal accuracy on the progress bar
                             robust=f"{100 * tot_adv / n:.1f}%")  # and the accuracy on tricked images
        # WHAT: Closes the progress bar.
        # HOW: Frees the terminal line.
        # WHY: Cleaner logs.
        # PROS & CONS: + Tidy. - None.
        # EXAMPLE: With leave=False the finished bar disappears and only the printed epoch line remains.
        pbar.close()  # close the progress bar
        # WHAT: Lowers the learning rate once per epoch.
        # HOW: Moves one step along the cosine curve.
        # WHY: Smooth convergence.
        # PROS & CONS: + Standard. - None.
        # EXAMPLE: lr 9.76e-5 after epoch 1.
        scheduler.step()  # lower the learning rate a little, once per epoch
        # WHAT: Switches to test mode before saving.
        # HOW: Disables training behaviour.
        # WHY: Saved models are used for evaluation.
        # PROS & CONS: + Safe. - None.
        # EXAMPLE: model.training is False.
        model.eval()  # test mode again
        # WHAT: Summary of the epoch.
        # HOW: Dictionary with epoch number, average loss, clean and robust training accuracy, learning rate and
        #      seconds.
        # WHY: Written to the log and used for the report's robust-generalisation numbers.
        # PROS & CONS: + Complete record. - Training accuracy is measured on augmented batches.
        # EXAMPLE: Epoch 10 ViT: robust 59.11%, 446 s.
        entry = {"epoch": epoch + 1, "train_loss": tot_loss / max(n, 1),  # summary of this epoch: epoch number and average loss (max(n, 1) avoids dividing by 0)
                 "train_clean_acc": 100 * tot_clean / max(n, 1), "train_robust_acc": 100 * tot_adv / max(n, 1),  # accuracy on normal images and on tricked images
                 "lr": optimizer.param_groups[0]["lr"], "seconds": time.time() - t0}  # learning rate used and time taken
        # WHAT: Prints the epoch result.
        # HOW: One line per epoch.
        # WHY: Keeps a readable log in Colab.
        # PROS & CONS: + Visible. - None.
        # EXAMPLE: [vit_adv] Epoch 10/10 | loss 0.52 | clean 99.85% | robust 59.11% | 446s
        print(f"[{tag}] Epoch {entry['epoch']}/{epochs} | loss {entry['train_loss']:.4f} | "  # print the epoch result: epoch number and loss
              f"clean {entry['train_clean_acc']:.2f}% | robust {entry['train_robust_acc']:.2f}% | "  # normal accuracy and tricked-image accuracy
              f"{entry['seconds']:.0f}s")  # seconds taken
        # WHAT: Saves the weights after every epoch.
        # HOW: Overwrites the file.
        # WHY: A disconnect loses at most one epoch.
        # PROS & CONS: + Safe. - Always the latest, not the best, epoch.
        # EXAMPLE: checkpoints/vit_adv.pth.
        save_checkpoint(model, ckpt_path)  # save the learned weights after every epoch
        # WHAT: Saves everything needed to resume.
        # HOW: Weights, optimiser, scheduler, scaler and epoch number.
        # WHY: Exact continuation.
        # PROS & CONS: + Resumable. - Large file.
        # EXAMPLE: vit_adv_state.pt.
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),  # also save everything needed to continue later: weights, weight-updating tool
                    "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),  # learning-rate plan, small-number protector
                    "epoch": epoch + 1}, state_path)  # and the epoch number
        # WHAT: Adds the epoch entry to the log.
        # HOW: Rewrites the JSON.
        # WHY: Training curves can be plotted later.
        # PROS & CONS: + Persistent. - None.
        # EXAMPLE: entries list grows to 10.
        update_history(hist_path, tag, entry=entry)  # add this epoch's result to the log
        # WHAT: Backs up to Google Drive.
        # HOW: Copies files if a Drive folder is given.
        # WHY: Colab local files vanish.
        # PROS & CONS: + Safe. - Slower epoch end.
        # EXAMPLE: Copies cnn_adv.pth.
        sync_to_drive(ckpt_dir, drive_dir)  # copy the saved files to Google Drive as a backup (only if a Drive folder is given)
        # WHAT: Optional GitHub backup every few epochs.
        # HOW: Pushes after epochs 5 (not the last).
        # WHY: Protects intermediate progress.
        # PROS & CONS: + Remote backup. - Needs a token.
        # EXAMPLE: epoch 5.
        if auto_push_every and (epoch + 1) % auto_push_every == 0 and (epoch + 1) < epochs:  # every few epochs, if upload is switched on
            push_fn(f"chore(train): {tag} epoch {epoch + 1}/{epochs} checkpoint")  # upload progress to GitHub
    # WHAT: Marks training as finished.
    # HOW: Sets completed=True.
    # WHY: Next run will skip retraining.
    # PROS & CONS: + Idempotent. - None.
    # EXAMPLE: completed: true.
    update_history(hist_path, tag, completed=True)  # mark in the log that training is finished
    # WHAT: Deletes the resume state.
    # HOW: The finished model is in the .pth file.
    # WHY: Frees disk space and prevents wrong resumes.
    # PROS & CONS: + Clean. - None.
    # EXAMPLE: vit_adv_state.pt removed.
    if os.path.exists(state_path):  # the "continue later" file is not needed any more
        os.remove(state_path)  # delete it
    # WHAT: Final backup.
    # HOW: Copies the finished files.
    # WHY: Secure result.
    # PROS & CONS: + Safe. - None.
    # EXAMPLE: cnn_adv.pth and training_history.json are copied to the Drive folder.
    sync_to_drive(ckpt_dir, drive_dir)  # final backup to Google Drive
    # WHAT: Leaves the model in test mode.
    # HOW: eval().
    # WHY: Callers evaluate it next.
    # PROS & CONS: + Safe. - None.
    # EXAMPLE: model.training is False when adversarial_finetune returns.
    model.eval()  # leave the model in test mode
    # WHAT: Returns the list of epoch summaries.
    # HOW: Reads them back from the log.
    # WHY: Lets scripts plot or inspect them.
    # PROS & CONS: + Convenient. - None.
    # EXAMPLE: 10 entries.
    return load_json(hist_path)[tag]["epochs"]  # give back the list of epoch results


# WHAT: Normal (non-adversarial) fine-tuning, used to prepare the ViT and the ResNet-18 for CIFAR-10.
# HOW: Standard training with cross-entropy, AdamW (lr 2e-4), cosine decay and mixed precision.
# WHY: A freshly attached 10-class head must learn CIFAR-10 before adversarial training starts.
# PROS & CONS: + Fast: no attack passes, so far cheaper than adversarial training. - No robustness at all.
# EXAMPLE: 5 epochs bring ViT-Tiny to 97.1% test accuracy.
def clean_finetune(model, train_loader, epochs=5, lr=2e-4, device=None, max_batches=0, desc="ViT-clean"):  # normal training (no tricked images); used once to prepare the ViT for CIFAR-10
    # WHAT: Picks the device for clean training.
    # HOW: Uses the model's own device when none is passed.
    # WHY: Prevents a mismatch between model and data devices.
    # PROS & CONS: + Safe default. - None.
    # EXAMPLE: clean_finetune(model, loader) trains on the model's current device.
    device = device or next(model.parameters()).device  # if no device was given, use the one the model is already on
    use_amp = device.type == "cuda"  # use faster half-precision maths only on a GPU
    # WHAT: Optimiser, scheduler and scaler for clean training.
    # HOW: AdamW lr 2e-4, cosine, GradScaler.
    # WHY: Same tools as adversarial training; larger learning rate because the head is new.
    # PROS & CONS: + Consistent. - None.
    # EXAMPLE: lr 0.0002.
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)  # the tool that updates the weights
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)  # slowly lowers the learning rate towards 0
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)  # stops very small numbers becoming 0 in half-precision maths
    # WHAT: Epoch loop of clean training.
    # HOW: Forward, loss, backward, step per batch.
    # WHY: Learn the 10 classes.
    # PROS & CONS: + Simple. - None.
    # EXAMPLE: 5 epochs.
    for epoch in range(epochs):  # one loop = one full pass over all training images
        model.train()  # learning mode
        tot = correct = n = 0  # running totals: loss, correct guesses, images seen
        pbar = tqdm(train_loader, desc=f"[{desc}] Epoch {epoch + 1}/{epochs}", leave=False)  # progress bar over the batches
        for b, (x, y) in enumerate(pbar):  # take one batch of images x and labels y
            if max_batches and b >= max_batches:  # only the quick test sets a limit
                break  # stop this epoch early
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)  # move the batch to the GPU/CPU
            optimizer.zero_grad(set_to_none=True)  # erase the update directions from the previous batch
            with _autocast(device, use_amp):  # faster maths if a GPU is used
                logits = model(x)  # 10 scores per image, normal images only
            # WHAT: Clean loss.
            # HOW: Cross-entropy only on normal images.
            # WHY: No attack here.
            # PROS & CONS: + Standard. - No robustness.
            # EXAMPLE: The loss falls over the 5 epochs as the new head learns.
            loss = F.cross_entropy(logits.float(), y)  # loss = how wrong the guesses are
            scaler.scale(loss).backward()  # work out how every weight should change
            scaler.step(optimizer)  # change the weights
            scaler.update()  # adjust the small-number protector
            tot += loss.item() * x.size(0)  # add this batch's loss to the running total
            correct += (logits.argmax(1) == y).sum().item()  # count right guesses (highest score = the guess)
            n += x.size(0)  # count the images seen so far
            pbar.set_postfix(loss=f"{tot / n:.3f}", acc=f"{100 * correct / n:.1f}%")  # show loss and accuracy on the progress bar
        pbar.close()  # close the progress bar
        scheduler.step()  # lower the learning rate a little
        # WHAT: Prints loss and accuracy after each epoch.
        # HOW: One log line.
        # WHY: Monitoring.
        # PROS & CONS: + Visible. - None.
        # EXAMPLE: [ViT-clean] Epoch 5/5 | loss <average loss> | acc <training accuracy>%
        print(f"[{desc}] Epoch {epoch + 1}/{epochs} | loss {tot / max(n, 1):.4f} | acc {100 * correct / max(n, 1):.2f}%")  # print this epoch's loss and accuracy
    # WHAT: Returns the trained model in test mode.
    # HOW: eval().
    # WHY: Ready for evaluation or copying.
    # PROS & CONS: + Safe. - None.
    # EXAMPLE: vit_clean = clean_finetune(vit_clean, loader, epochs=5, device=device).
    return model.eval()  # give back the trained model in test mode

In [ ]:
%%writefile src/evaluate.py
"""Evaluation: measure how accurate each model is, with and without attacks."""
# WHAT: Imports file helpers, pandas tables, PyTorch and the two evaluation attacks.
# HOW: pandas builds the result tables that become CSV and Markdown.
# WHY: Evaluation produces tables.
# PROS & CONS: + Standard. - None.
# EXAMPLE: pd.DataFrame(rows).
import os  # work with file and folder paths

import pandas as pd  # tables (like Excel) in Python
import torch  # main library for deep learning

from .attacks import fgsm_attack, pgd_attack  # the two attacks used for testing

# WHAT: Attack strengths and model order used everywhere.
# HOW: eps values in 1/255 units; 0 means no attack.
# WHY: Same grid for every model, covering clean to twice the training budget.
# PROS & CONS: + Comparable curves. - Only 6 points per curve.
# EXAMPLE: [0, 1, 2, 4, 8, 16].
EPS_LIST = [0, 1, 2, 4, 8, 16]  # attack strengths to test, in 1/255 units (0 = no attack = normal accuracy)
MODEL_ORDER = ["CNN-Clean", "ViT-Clean", "CNN-AdvTrained", "ViT-AdvTrained"]  # the 4 models, in the order we show them


# WHAT: Counts correct predictions without tracking gradients.
# HOW: argmax of the 10 scores is the prediction.
# WHY: Fast and memory-light.
# PROS & CONS: + Efficient. - None.
# EXAMPLE: _correct(model, x, y) returns an integer.
@torch.no_grad()  # do not keep learning information here (faster, uses less memory)
def _correct(model, x, y):  # counts how many images the model guesses right
    return (model(x).argmax(1) == y).sum().item()  # the highest of the 10 scores is the guess; count guesses equal to the true label


# WHAT: Accuracy in percent on a set of images.
# HOW: Processes in batches of 100, sums correct predictions.
# WHY: Batches keep memory low.
# PROS & CONS: + Works for any size. - None.
# EXAMPLE: accuracy(model, X, Y, device) = 91.5.
def accuracy(model, X, Y, device, batch_size=100):  # accuracy in % of the model on images X with true labels Y
    model.eval()  # test mode
    c = 0  # number of right guesses so far
    for i in range(0, len(X), batch_size):  # go through the images in groups of 100 (so memory does not run out)
        x, y = X[i:i + batch_size].to(device), Y[i:i + batch_size].to(device)  # take one group and move it to the GPU/CPU
        c += _correct(model, x, y)  # add the right guesses of this group
    return 100.0 * c / len(X)  # right guesses divided by total images, as a percentage


# WHAT: Returns the attack function for a name and eps.
# HOW: FGSM or PGD-10 (alpha = max(eps/4, 1/255), random start).
# WHY: One place defines the evaluation attacks.
# PROS & CONS: + Consistent settings. - Only two attacks.
# EXAMPLE: _attack_fn('PGD-10', 8/255, 10).
def _attack_fn(name, eps, pgd_steps):  # gives back the attack function with the test settings filled in
    if name == "FGSM":  # the one-step attack
        return lambda m, x, y: fgsm_attack(m, x, y, eps)  # attack with strength eps
    if name == "PGD-10":  # the strong many-step attack
        return lambda m, x, y: pgd_attack(m, x, y, eps, alpha=max(eps / 4, 1 / 255), steps=pgd_steps,  # small steps of size eps/4 (at least 1/255), repeated pgd_steps times
                                          random_start=True)  # begin from a randomly shaken image
    raise ValueError(name)  # any other name is a mistake, so show an error


# WHAT: Accuracy of every model under every attack at every eps.
# HOW: For each model, attack and eps: fix the seed, attack each batch with the model itself (white-box), count
#      correct predictions.
# WHY: Produces the accuracy-versus-epsilon curves requested by the mentor.
# PROS & CONS: + Complete and repeatable. - Cost grows with models, eps values and PGD steps.
# EXAMPLE: Result: ViT-AdvTrained, PGD-10, eps = 8 -> 42.2%.
def evaluate_epsilon_sweep(models, X, Y, device, eps_list=EPS_LIST, pgd_steps=10, batch_size=100, seed=42):  # test every model against every attack at every strength
    # WHAT: Result list.
    # HOW: One dictionary per combination.
    # WHY: Becomes the table.
    # PROS & CONS: + Simple. - None.
    # EXAMPLE: []
    rows = []  # one row of results for each combination
    # WHAT: Loops over the 4 models.
    # HOW: Sets test mode.
    # WHY: Attack and evaluate each.
    # PROS & CONS: + Clear. - None.
    # EXAMPLE: CNN-Clean first.
    for name, model in models.items():  # go through the 4 models
        model.eval()  # test mode
        # WHAT: Loops over attacks and budgets.
        # HOW: Nested loops.
        # WHY: Full grid.
        # PROS & CONS: + Complete. - 48 evaluations.
        # EXAMPLE: FGSM then PGD-10.
        for attack in ("FGSM", "PGD-10"):  # go through both attacks
            for e in eps_list:  # go through every attack strength
                # WHAT: Fixes the random numbers before each cell of the grid.
                # HOW: Seed 42 each time.
                # WHY: Random starts become repeatable.
                # PROS & CONS: + Reproducible. - Same noise pattern for every cell.
                # EXAMPLE: Seed 42 is set before every (model, attack, eps) cell, so a re-run gives the same
                #          numbers.
                torch.manual_seed(seed)  # fixed starting number so the random parts are the same every run
                # WHAT: Converts eps to pixel units.
                # HOW: 8 becomes 0.0314.
                # WHY: Pixels live in [0, 1].
                # PROS & CONS: + Correct scale. - None.
                # EXAMPLE: 8 / 255 = 0.03137.
                eps = e / 255.0  # change e.g. 8 into 8/255, because pixel values are between 0 and 1
                # WHAT: Gets the attack function for this attack name and eps.
                # HOW: Calls _attack_fn, which returns a function (model, x, y) -> x_adv.
                # WHY: The batch loop can then call fn(model, x, y) without knowing which attack it is.
                # PROS & CONS: + One loop serves both attacks. - None.
                # EXAMPLE: fn = _attack_fn('PGD-10', 8/255, 10).
                fn = _attack_fn(attack, eps, pgd_steps)  # get the attack function for this attack and strength
                correct = 0  # right guesses so far
                # WHAT: Attacks the 1000 images in batches of 100.
                # HOW: x_adv = attack(model, x, y), then count the model's correct answers on x_adv.
                # WHY: Memory limits.
                # PROS & CONS: + Works on small GPUs. - None.
                # EXAMPLE: 10 batches.
                for i in range(0, len(X), batch_size):  # go through the test images in groups
                    x, y = X[i:i + batch_size].to(device), Y[i:i + batch_size].to(device)  # take one group and move it to the GPU/CPU
                    x_adv = fn(model, x, y)  # step 1: trick the images (the attacker uses this same model)
                    correct += _correct(model, x_adv, y)  # step 2: count how many tricked images are still guessed right
                # WHAT: Accuracy in percent.
                # HOW: correct / N.
                # WHY: Report format.
                # PROS & CONS: + Direct. - Standard error up to 1.6 points for N = 1000.
                # EXAMPLE: 422 correct out of 1000 = 42.2%.
                acc = 100.0 * correct / len(X)  # accuracy in percent
                # WHAT: Saves and prints the result.
                # HOW: Dictionary row plus a console line.
                # WHY: Feeds the CSV and plots.
                # PROS & CONS: + Visible. - None.
                # EXAMPLE: [eval] ViT-AdvTrained PGD-10 eps=8/255 -> 42.2%
                rows.append({"model": name, "attack": attack, "eps_255": e, "accuracy": acc})  # save this result as one row
                print(f"[eval] {name:15s} {attack:6s} eps={e:2d}/255 -> {acc:5.1f}%")  # show the result on screen
    # WHAT: Returns all results as a table.
    # HOW: pandas DataFrame.
    # WHY: Easy to save and plot.
    # PROS & CONS: + Convenient. - None.
    # EXAMPLE: 48 rows.
    return pd.DataFrame(rows)  # turn all rows into one table


# WHAT: Transfer test: craft PGD-10 examples on a source model, test them on all target models.
# HOW: 4x4 matrix: row = source, column = target; the diagonal is white-box.
# WHY: Shows whether attacks transfer across architectures and from clean to defended models.
# PROS & CONS: + Black-box view of robustness. - Only eps = 8/255 and PGD-10.
# EXAMPLE: CNN-Clean to ViT-Clean: 85.4% remains.
def evaluate_transferability(models, X, Y, device, eps=8 / 255, pgd_steps=10, batch_size=100, seed=42):  # does a trick made on one model also fool another model?
    # WHAT: Empty result matrix.
    # HOW: pandas DataFrame with model names on both axes.
    # WHY: Filled in the loop.
    # PROS & CONS: + Clear. - None.
    # EXAMPLE: 4x4.
    names = list(models.keys())  # the 4 model names
    mat = pd.DataFrame(index=names, columns=names, dtype=float)  # empty 4x4 table: rows = model that made the trick, columns = model being tested
    # WHAT: Generates adversarial images from each source model.
    # HOW: PGD-10 on every batch; moved to CPU to save GPU memory.
    # WHY: These images are reused for all targets.
    # PROS & CONS: + One attack per source. - Stores 1000 adversarial images on the CPU.
    # EXAMPLE: (1000, 3, 32, 32).
    for src in names:  # take each model in turn as the one that makes the tricked images
        torch.manual_seed(seed)  # fixed starting number for repeatable results
        models[src].eval()  # test mode
        adv_batches = []  # will hold the tricked images
        for i in range(0, len(X), batch_size):  # go through the test images in groups
            x, y = X[i:i + batch_size].to(device), Y[i:i + batch_size].to(device)  # take one group and move it to the GPU/CPU
            adv_batches.append(pgd_attack(models[src], x, y, eps, alpha=max(eps / 4, 1 / 255),  # trick the group using the PGD attack on the source model
                                          steps=pgd_steps, random_start=True).cpu())  # many small steps, random start; move result to the CPU to save GPU memory
        X_adv = torch.cat(adv_batches)  # join all groups into one set of tricked images
        # WHAT: Tests every target model on the same adversarial images.
        # HOW: accuracy(target, X_adv, Y).
        # WHY: Fills one matrix row.
        # PROS & CONS: + Fair comparison. - None.
        # EXAMPLE: Row CNN-Clean: 0.0, 85.4, 69.2, 87.1.
        for tgt in names:  # test the same tricked images on every model
            mat.loc[src, tgt] = accuracy(models[tgt], X_adv, Y, device, batch_size)  # store that model's accuracy in the table
        print(f"[transfer] source={src:15s} -> " + ", ".join(f"{t}: {mat.loc[src, t]:.1f}%" for t in names))  # show this row of the table on screen
    # WHAT: Label and result.
    # HOW: Names the table corner and returns it.
    # WHY: Readable CSV.
    # PROS & CONS: + Clear. - None.
    # EXAMPLE: 'source \ target'.
    mat.index.name = "source \\ target"  # label the table corner: rows = source, columns = target
    return mat  # give back the finished table


# WHAT: Converts a table to Markdown text.
# HOW: Builds header, divider and one line per row with 1-decimal numbers.
# WHY: GitHub renders Markdown tables.
# PROS & CONS: + No extra library. - Very small formatter.
# EXAMPLE: | CNN-Clean | 91.5 | ...
def to_markdown(df, floatfmt="{:.1f}"):  # turns a table into text that GitHub can show as a table
    cols = [df.index.name or ""] + [str(c) for c in df.columns]  # the column titles
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join([" --- "] * len(cols)) + "|"]  # title line plus the "---" divider line
    for idx, row in df.iterrows():  # go through each row of the table
        cells = [str(idx)] + [floatfmt.format(v) if isinstance(v, float) else str(v) for v in row.values]  # row name, then each number with 1 decimal place
        lines.append("| " + " | ".join(cells) + " |")  # add this row as a text line
    return "\n".join(lines)  # join all lines into one block of text


# WHAT: Writes all result files.
# HOW: results_summary.csv, results_summary.md and transferability.csv.
# WHY: Tracked in git so the tables are visible on GitHub.
# PROS & CONS: + Reproducible outputs. - Overwrites previous files.
# EXAMPLE: outputs/results_summary.md.
def export_results(sweep_df, transfer_df, out_dir, note=""):  # saves all results as files (CSV and Markdown)
    os.makedirs(out_dir, exist_ok=True)  # create the output folder if it is missing
    sweep_df.to_csv(os.path.join(out_dir, "results_summary.csv"), index=False)  # save the long results list as a CSV file
    wide = sweep_df.pivot_table(index=["attack", "model"], columns="eps_255", values="accuracy", sort=False)  # reshape: one row per attack+model, one column per strength
    wide.columns = [f"eps={c}/255" for c in wide.columns]  # give the columns readable names
    wide.index = [f"{a}, {m}" for a, m in wide.index]  # join attack and model into one row name
    wide.index.name = "attack, model"  # name of the row-title column
    md = ["# Results summary (accuracy %, white-box)", ""]  # first lines of the Markdown report
    if note:  # if a note was given
        md += [f"> {note}", ""]  # add it near the top
    md += [to_markdown(wide), "", "## Transferability: PGD-10 at eps=8/255 (rows = source, cols = target; accuracy %)", "",  # add the main table, then the heading of the second table
           to_markdown(transfer_df), ""]  # add the second table (which model's tricks fool which model)
    with open(os.path.join(out_dir, "results_summary.md"), "w", encoding="utf-8") as f:  # open the Markdown file for writing
        f.write("\n".join(md))  # write the report into it
    transfer_df.to_csv(os.path.join(out_dir, "transferability.csv"))  # save the second table as a CSV file
    print("\n".join(md))  # also show the report on screen

In [ ]:
%%writefile src/visualize.py
"""Pictures: draws the 4 result graphs and saves them as image files."""
# WHAT: Imports the path helper used to create the output folder.
# HOW: os.makedirs and os.path.join build the PNG file paths.
# WHY: The plots are saved as files, never shown in a window.
# PROS & CONS: + Standard library. - None.
# EXAMPLE: os.path.join('outputs', 'acc_vs_epsilon.png').
import os  # work with file and folder paths

# WHAT: Imports matplotlib and selects the 'Agg' backend.
# HOW: Agg renders straight into image files with no display window.
# WHY: Colab, Kaggle and servers have no screen; a GUI backend would crash.
# PROS & CONS: + Works anywhere. - No interactive zoom.
# EXAMPLE: fig.savefig('x.png') works on a headless machine.
import matplotlib  # drawing library
matplotlib.use("Agg")  # draw into files only (no pop-up window), so it also works on a server
# WHAT: Imports plotting, array, colour-table and PyTorch libraries.
# HOW: pyplot draws figures, numpy positions bars, seaborn draws the heat map, torch runs the attack for the
#      example figure.
# WHY: Each library covers one part of the figures.
# PROS & CONS: + Mature tools. - seaborn is only needed for one figure.
# EXAMPLE: sns.heatmap(matrix, annot=True).
import matplotlib.pyplot as plt  # the drawing commands
import numpy as np  # fast number lists
import seaborn as sns  # nicer-looking graphs (we use it for the colour table)
import torch  # main library for deep learning

# WHAT: Imports the PGD attack and the class names.
# HOW: Relative imports from the same package.
# WHY: The example figure creates fresh adversarial images; the class names label them.
# PROS & CONS: + Reuses tested code. - None.
# EXAMPLE: CIFAR10_CLASSES[3] == 'cat'.
from .attacks import pgd_attack  # the strong attack, used to make the example pictures
from .models import CIFAR10_CLASSES  # names of the 10 picture types

# WHAT: Picture sharpness and model colours.
# HOW: 300 dots per inch; blue for the CNN, red for the ViT.
# WHY: 300 DPI is print quality for the report; fixed colours make the figures easy to read.
# PROS & CONS: + Consistent look. - Larger files (200-350 KB per PNG).
# EXAMPLE: COLORS['ViT'] == '#d62728'.
DPI = 300  # picture sharpness (300 = print quality)
COLORS = {"CNN": "#1f77b4", "ViT": "#d62728"}  # blue for CNN, red for ViT


# WHAT: Chooses colour, line style and marker for a model name.
# HOW: The part before '-' gives the architecture; 'Adv' in the name gives a dashed line with squares.
# WHY: Solid circles = clean model, dashed squares = adversarially trained model.
# PROS & CONS: + Visual meaning is consistent in all plots. - Relies on the naming convention.
# EXAMPLE: _style('ViT-AdvTrained') returns ('#d62728', '--', 's').
def _style(name):  # picks line colour, line style and marker shape for a model name
    # WHAT: Extracts the architecture from the model name.
    # HOW: split('-')[0].
    # WHY: Used as a key into COLORS.
    # PROS & CONS: + Simple. - Breaks if names change.
    # EXAMPLE: 'CNN-Clean' becomes 'CNN'.
    arch = name.split("-")[0]  # first part of the name: "CNN" or "ViT"
    # WHAT: Returns the style triple.
    # HOW: colour, linestyle, marker.
    # WHY: Consumed by ax.plot.
    # PROS & CONS: + Clear. - None.
    # EXAMPLE: ('#1f77b4', '-', 'o').
    return COLORS[arch], ("--" if "Adv" in name else "-"), ("s" if "Adv" in name else "o")  # colour by type; dashed line + square for defended models, solid line + circle for normal ones


# WHAT: Adds a small grey note at the bottom of a figure.
# HOW: fig.text at the bottom centre when a tag is given.
# WHY: Marks quick-test figures as 'not a result' so they cannot be mistaken for real ones.
# PROS & CONS: + Prevents misuse. - None.
# EXAMPLE: 'QUICK-TEST smoke run ... not a result'.
def _tag(fig, tag):  # writes a small grey note at the bottom of a graph
    if tag:  # only if a note was given (used to mark quick-test graphs as "not a real result")
        fig.text(0.5, 0.005, tag, ha="center", va="bottom", fontsize=8, color="gray")  # place the note in the bottom middle


# WHAT: Figure 1: accuracy versus attack strength, FGSM on the left and PGD-10 on the right.
# HOW: For every model draw one line (eps on x, accuracy on y). Solid = clean, dashed = adversarially trained.
# WHY: This is the accuracy-versus-epsilon curve the mentor asked for.
# PROS & CONS: + Shows all four models in one view. - No error bars (standard error up to 1.6 points).
# EXAMPLE: outputs/acc_vs_epsilon.png.
def plot_acc_vs_epsilon(df, path, tag=""):  # graph 1: accuracy as the attack gets stronger (FGSM on the left, PGD-10 on the right)
    # WHAT: Creates two side-by-side panels with a shared y axis.
    # HOW: 1 row, 2 columns, 12 by 4.6 inches.
    # WHY: Sharing the axis makes FGSM and PGD directly comparable.
    # PROS & CONS: + Easy comparison. - Wide figure.
    # EXAMPLE: axes[0] = FGSM, axes[1] = PGD-10.
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)  # a canvas with two graphs side by side that share the up-down scale
    # WHAT: Loops over the two attacks.
    # HOW: One panel per attack.
    # WHY: Same drawing code for both.
    # PROS & CONS: + No duplication. - None.
    # EXAMPLE: ('FGSM', 'PGD-10').
    for ax, attack in zip(axes, ("FGSM", "PGD-10")):  # one graph for each attack
        # WHAT: Loops over the four models.
        # HOW: One line per model.
        # WHY: All models in each panel.
        # PROS & CONS: + Complete. - Four lines can overlap at eps = 0.
        # EXAMPLE: 'CNN-Clean', 'ViT-Clean', ...
        for name in df["model"].unique():  # one line for each model
            # WHAT: Selects the rows for this model and attack, sorted by eps.
            # HOW: Boolean filter, then sort_values('eps_255').
            # WHY: Lines must be drawn from small to large eps.
            # PROS & CONS: + Correct curve order. - None.
            # EXAMPLE: 6 rows (eps 0, 1, 2, 4, 8, 16).
            d = df[(df.model == name) & (df.attack == attack)].sort_values("eps_255")  # keep this model's rows for this attack, ordered by strength
            # WHAT: Gets the style for this model.
            # HOW: Calls _style.
            # WHY: Consistent colours.
            # PROS & CONS: + Consistent. - None.
            # EXAMPLE: ('#1f77b4', '-', 'o').
            c, ls, mk = _style(name)  # colour, line style and marker for this model
            # WHAT: Draws the line.
            # HOW: x = eps in /255 units, y = accuracy in percent, with markers.
            # WHY: Shows how accuracy collapses as the attack gets stronger.
            # PROS & CONS: + Clear trend. - Markers hide values at eps = 0 where all models start high.
            # EXAMPLE: (8, 42.2) is one point of the ViT-AdvTrained PGD-10 curve.
            ax.plot(d.eps_255, d.accuracy, color=c, linestyle=ls, marker=mk, linewidth=2, label=name)  # draw the line: attack strength across, accuracy up
        # WHAT: Titles, axis label, tick marks and grid of each panel.
        # HOW: x ticks sit exactly at the tested eps values.
        # WHY: Readers can find each tested budget.
        # PROS & CONS: + Readable. - Uneven spacing near 0.
        # EXAMPLE: 'FGSM attack'.
        ax.set_title(f"{attack} attack")  # graph title
        ax.set_xlabel(r"$\epsilon$ (in /255 units)")  # label under the graph: attack strength
        ax.set_xticks(sorted(df.eps_255.unique()))  # show a mark for each strength we tested
        ax.grid(alpha=0.3)  # faint grid lines
    # WHAT: Y axis label and range on the left panel.
    # HOW: Range fixed to 0..100 percent.
    # WHY: Honest, comparable scale.
    # PROS & CONS: + No exaggeration. - Small differences are hard to see.
    # EXAMPLE: 'Accuracy (%)'.
    axes[0].set_ylabel("Accuracy (%)")  # label on the left side
    axes[0].set_ylim(0, 100)  # the up-down scale goes from 0 to 100 percent
    # WHAT: Legend with the four model names.
    # HOW: Placed upper right of the PGD panel.
    # WHY: Explains the colours and line styles.
    # PROS & CONS: + Needed. - May cover points.
    # EXAMPLE: 'CNN-Clean', 'ViT-Clean', ...
    axes[1].legend(loc="upper right")  # box that says which line is which model
    # WHAT: Overall title.
    # HOW: States that solid = clean and dashed = adversarially fine-tuned.
    # WHY: Self-explanatory figure.
    # PROS & CONS: + Standalone. - Long title.
    # EXAMPLE: Accuracy vs. perturbation budget on CIFAR-10 ...
    fig.suptitle("Accuracy vs. perturbation budget on CIFAR-10 (solid = clean, dashed = adversarially fine-tuned)")  # main title
    # WHAT: Finishes and saves the figure.
    # HOW: Fit the layout, add the optional note, write the PNG at 300 DPI, close the figure.
    # WHY: close() frees memory; tight_layout avoids clipped labels.
    # PROS & CONS: + Clean output. - None.
    # EXAMPLE: outputs/acc_vs_epsilon.png (3600x1380 pixels).
    fig.tight_layout(rect=(0, 0.03, 1, 1))  # fit everything neatly, leaving room at the bottom
    _tag(fig, tag)  # add the small grey note if there is one
    fig.savefig(path, dpi=DPI)  # save the graph as an image file
    plt.close(fig)  # release the memory


# WHAT: Looks up one accuracy value.
# HOW: Filters the result table by model, attack and eps and returns the first value as a float.
# WHY: Used by the trade-off figure.
# PROS & CONS: + Simple. - Raises an error if the row is missing.
# EXAMPLE: _acc(df, 'ViT-Clean', 'PGD-10', 8) == 0.0.
def _acc(df, model, attack, eps):  # looks up one accuracy value in the results table
    # WHAT: The filter itself.
    # HOW: Three equality conditions combined with &.
    # WHY: Exactly one row matches.
    # PROS & CONS: + Precise. - None.
    # EXAMPLE: 0.0
    return float(df[(df.model == model) & (df.attack == attack) & (df.eps_255 == eps)].accuracy.iloc[0])  # find the row for this model, attack and strength; take its accuracy


# WHAT: Figure 2: what adversarial training costs and gains.
# HOW: Grouped bars (before / after) for clean, FGSM (8/255) and PGD-10 (8/255) accuracy, with the change in
#      percentage points written above.
# WHY: Answers RQ2: cost of adversarial training on clean accuracy versus robustness gain.
# PROS & CONS: + Cost and gain visible together. - Only the 8/255 budget.
# EXAMPLE: CNN: -3.1 pp clean, +18.3 pp PGD-10.
def plot_tradeoff(df, path, tag=""):  # graph 2: what defence training costs (normal accuracy) and gains (accuracy under attack)
    # WHAT: The three measurements shown.
    # HOW: (label, attack, eps). FGSM at eps 0 equals clean accuracy.
    # WHY: A clean number needs no extra evaluation.
    # PROS & CONS: + Reuses existing data. - Relies on FGSM(0) == clean.
    # EXAMPLE: ('Clean (eps=0)', 'FGSM', 0).
    metrics = [("Clean (eps=0)", "FGSM", 0), ("FGSM (eps=8/255)", "FGSM", 8), ("PGD-10 (eps=8/255)", "PGD-10", 8)]  # three tests: no attack, FGSM, PGD-10 (FGSM with eps=0 is just normal accuracy)
    # WHAT: Two panels: CNN and ViT.
    # HOW: Shared y axis.
    # WHY: Same scale for both architectures.
    # PROS & CONS: + Comparable. - None.
    # EXAMPLE: axes[0] = CNN.
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)  # a canvas with two graphs side by side: CNN and ViT
    # WHAT: Loops over the two architectures.
    # HOW: One panel each.
    # WHY: Same drawing code.
    # PROS & CONS: + No duplication. - None.
    # EXAMPLE: ('CNN', 'ViT').
    for ax, arch in zip(axes, ("CNN", "ViT")):  # one graph for each model type
        # WHAT: Collects the accuracies before and after defence training.
        # HOW: Lookups of the Clean and AdvTrained models.
        # WHY: Bars compare the two.
        # PROS & CONS: + Clear. - None.
        # EXAMPLE: CNN before: [91.5, 11.9, 0.0].
        before = [_acc(df, f"{arch}-Clean", a, e) for _, a, e in metrics]  # accuracies of the normal model
        after = [_acc(df, f"{arch}-AdvTrained", a, e) for _, a, e in metrics]  # accuracies of the defended model
        # WHAT: Bar positions and width.
        # HOW: Positions 0, 1, 2; each bar 0.36 wide.
        # WHY: Two bars side by side per group.
        # PROS & CONS: + Neat. - None.
        # EXAMPLE: [0, 1, 2].
        xs = np.arange(len(metrics))  # positions of the bar groups: 0, 1, 2
        w = 0.36  # width of one bar
        # WHAT: Draws the 'before' bars.
        # HOW: Light colour, shifted left.
        # WHY: Light = clean model.
        # PROS & CONS: + Clear. - None.
        # EXAMPLE: Light blue for the CNN.
        ax.bar(xs - w / 2, before, w, label="Before (clean model)", color="#9ecae1" if arch == "CNN" else "#fcae91")  # light bars = before defence training
        # WHAT: Draws the 'after' bars.
        # HOW: Strong colour, shifted right.
        # WHY: Strong colour = defended model.
        # PROS & CONS: + Clear. - None.
        # EXAMPLE: Dark blue for the CNN.
        ax.bar(xs + w / 2, after, w, label="After 10-epoch adv. fine-tuning", color=COLORS[arch])  # strong-colour bars = after defence training
        # WHAT: Annotates each group.
        # HOW: Writes both values and the change.
        # WHY: Numbers are easier to read than bar heights.
        # PROS & CONS: + Precise. - Text can overlap in a crowded plot.
        # EXAMPLE: 'Delta Acc = +42.2 pp' style label.
        for i in range(len(metrics)):  # go through the three tests
            # WHAT: Change in percentage points.
            # HOW: Subtract before from after.
            # WHY: Quantifies the trade-off.
            # PROS & CONS: + Clear. - Absolute points, not relative change.
            # EXAMPLE: 94.1 - 97.1 = -3.0.
            delta = after[i] - before[i]  # change in accuracy (in percentage points)
            # WHAT: Writes the two values above the bars.
            # HOW: One decimal place.
            # WHY: Reading exact numbers.
            # PROS & CONS: + Precise. - None.
            # EXAMPLE: '97.1' and '94.1'.
            ax.text(xs[i] - w / 2, before[i] + 1, f"{before[i]:.1f}", ha="center", fontsize=8)  # write the "before" number above its bar
            ax.text(xs[i] + w / 2, after[i] + 1, f"{after[i]:.1f}", ha="center", fontsize=8)  # write the "after" number above its bar
            # WHAT: Chooses the label for the change.
            # HOW: Clean change for the first group, robust change for the others.
            # WHY: Different meaning.
            # PROS & CONS: + Clear. - None.
            # EXAMPLE: 'Delta Acc_clean' vs 'Delta Acc_robust'.
            label = r"$\Delta$Acc$_{clean}$" if i == 0 else r"$\Delta$Acc$_{robust}$"  # name of the change: normal-accuracy change or attacked-accuracy change
            # WHAT: Writes the change above each group.
            # HOW: Green if accuracy rose, red if it fell.
            # WHY: Colour tells the direction at a glance.
            # PROS & CONS: + Quick reading. - None.
            # EXAMPLE: Green '+42.2 pp'.
            ax.text(xs[i], max(before[i], after[i]) + 8, f"{label} = {delta:+.1f} pp", ha="center", fontsize=9,  # write the change above the bars
                    fontweight="bold", color="darkgreen" if delta >= 0 else "darkred")  # bold; green if accuracy went up, red if it went down
        # WHAT: Group names on the x axis.
        # HOW: One label per group.
        # WHY: Shows what is measured.
        # PROS & CONS: + Clear. - None.
        # EXAMPLE: 'PGD-10 (eps=8/255)'.
        ax.set_xticks(xs)  # one mark per bar group
        ax.set_xticklabels([m[0] for m in metrics])  # name each group
        # WHAT: Title and y range of each panel.
        # HOW: Range 0 to 115 leaves room for text.
        # WHY: Prevents text clipping.
        # PROS & CONS: + Readable. - Axis above 100.
        # EXAMPLE: 'ViT: cost of adversarial training vs. robustness gain'.
        ax.set_title(f"{arch}: cost of adversarial training vs. robustness gain")  # graph title
        ax.set_ylim(0, 115)  # the up-down scale (a bit above 100 to leave room for the text)
        ax.grid(axis="y", alpha=0.3)  # faint horizontal grid lines
        ax.legend(loc="upper right", fontsize=8)  # box that says which bar is which
    # WHAT: Y label and finishing steps.
    # HOW: Finishes and saves the figure.
    # WHY: tight_layout fits the labels, _tag writes the optional note, savefig writes the PNG at 300 DPI, close
    #      frees the figure.
    # PROS & CONS: + Clean output. - None.
    # EXAMPLE: outputs/clean_vs_robust_tradeoff.png.
    axes[0].set_ylabel("Accuracy (%)")  # label on the left side
    fig.tight_layout(rect=(0, 0.03, 1, 1))  # fit everything neatly
    _tag(fig, tag)  # add the small grey note if there is one
    fig.savefig(path, dpi=DPI)  # save the graph as an image file
    plt.close(fig)  # release the memory


# WHAT: Figure 3: heat map of the 4x4 transferability matrix.
# HOW: Colour from red (0%, strong attack) to green (100%, attack fails), with the number in each cell.
# WHY: Shows at a glance which source model fools which target model.
# PROS & CONS: + Easy pattern spotting. - Cell colours exaggerate small differences.
# EXAMPLE: Cell (CNN-Clean, ViT-Clean) = 85.4.
def plot_transferability(matrix, path, tag=""):  # graph 3: colour table showing which model's tricks fool which model
    # WHAT: One canvas.
    # HOW: 7.5 by 6 inches.
    # WHY: One heat map.
    # PROS & CONS: + Simple. - None.
    # EXAMPLE: One 7.5 x 6 inch canvas holds the whole 4x4 heat map.
    fig, ax = plt.subplots(figsize=(7.5, 6))  # a canvas for one graph
    # WHAT: Draws the table as coloured squares.
    # HOW: annot=True writes values; colour range fixed to 0..100.
    # WHY: Fixed range keeps colours comparable.
    # PROS & CONS: + Quick to read. - Needs seaborn.
    # EXAMPLE: 0.0 is dark red.
    sns.heatmap(matrix.astype(float), annot=True, fmt=".1f", cmap="RdYlGn", vmin=0, vmax=100, ax=ax,  # coloured squares with the number written inside; red = low accuracy, green = high
                cbar_kws={"label": "Target accuracy (%) - lower = stronger attack"})  # label of the colour bar
    # WHAT: Axis labels and title.
    # HOW: Rows = source (crafted on), columns = target (evaluated on).
    # WHY: A reader must know the direction.
    # PROS & CONS: + Clear. - Long label.
    # EXAMPLE: 'Source model (adversarial examples crafted on)'.
    ax.set_xlabel("Target model (evaluated on)")  # label under the table: the model being tested
    ax.set_ylabel("Source model (adversarial examples crafted on)")  # label on the left: the model that made the tricked images
    ax.set_title(r"Transferability of PGD-10 examples ($\epsilon$ = 8/255); diagonal = white-box")  # title; the diagonal is where the same model is both attacked and tested
    # WHAT: Tilts the x labels.
    # HOW: 20 degrees, right aligned.
    # WHY: Long names would overlap.
    # PROS & CONS: + Readable. - Slight crowding of the y label.
    # EXAMPLE: 'CNN-AdvTrained' is written at a 20-degree angle.
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right")  # tilt the names under the table so they do not overlap
    # WHAT: Layout, optional note, save, close.
    # HOW: tight_layout fits the labels, _tag writes the optional note, savefig writes the PNG at 300 DPI, close
    #      frees the figure.
    # WHY: Clipped labels and leaked figures are the usual matplotlib problems in long runs; the note stops
    #      quick-test images being mistaken for results.
    # PROS & CONS: + Clean output. - None.
    # EXAMPLE: outputs/transferability_heatmap.png.
    fig.tight_layout(rect=(0, 0.03, 1, 1))  # fit everything neatly
    _tag(fig, tag)  # add the small grey note if there is one
    fig.savefig(path, dpi=DPI)  # save the graph as an image file
    plt.close(fig)  # release the memory


# WHAT: Figure 4: real pictures, the (magnified) noise and the adversarial pictures.
# HOW: Picks one correctly classified test image per class, attacks them with PGD-10 on the source model, and
#      shows what all four models predict.
# WHY: Shows that the attack is invisible but effective.
# PROS & CONS: + Intuitive evidence. - Only 4 examples; an illustration, not evidence.
# EXAMPLE: frog classified as dog by CNN-Clean.
def plot_adversarial_examples(models, X, Y, device, path, eps=8 / 255, source="CNN-Clean", n=4,  # graph 4: shows real pictures, the added noise, and the tricked pictures
                              pgd_steps=10, tag=""):  # pgd_steps = how many small steps the attack takes
    # WHAT: Chooses the model that creates the adversarial images.
    # HOW: Default CNN-Clean, in test mode.
    # WHY: A single source keeps the figure simple.
    # PROS & CONS: + Simple. - Examples only show attacks on the CNN.
    # EXAMPLE: source='CNN-Clean'.
    src = models[source].eval()  # the model that makes the tricked images, in test mode
    # WHAT: Bookkeeping for picking pictures.
    # HOW: chosen = indices; seen = classes already picked.
    # WHY: Pick one picture per class.
    # PROS & CONS: + Variety. - None.
    # EXAMPLE: [] and set().
    chosen, seen = [], set()  # chosen = picked picture positions; seen = classes already picked
    # WHAT: Searches for suitable pictures without gradients.
    # HOW: Loop over the test images.
    # WHY: Only predictions needed.
    # PROS & CONS: + Fast. - None.
    # EXAMPLE: Predicting 1000 images without gradients builds no computation graph.
    with torch.no_grad():  # no learning information needed here
        # WHAT: Goes through the test images.
        # HOW: Stops when n pictures are chosen.
        # WHY: Finds n different classes.
        # PROS & CONS: + Cheap. - None.
        # EXAMPLE: Stops after 4 pictures.
        for i in range(len(X)):  # go through the test pictures
            # WHAT: True label of this picture.
            # HOW: Convert tensor to int.
            # WHY: Needed for the class check.
            # PROS & CONS: + Simple. - None.
            # EXAMPLE: 6 for frog.
            y = int(Y[i])  # the true label of this picture
            # WHAT: Skips classes already picked.
            # HOW: Set membership.
            # WHY: One picture per class.
            # PROS & CONS: + Variety. - None.
            # EXAMPLE: continue
            if y in seen:  # we already have a picture of this class
                continue  # skip it
            # WHAT: Keeps only pictures the model classifies correctly.
            # HOW: argmax equals the true label.
            # WHY: An attack on an already wrong picture shows nothing.
            # PROS & CONS: + Meaningful examples. - Cherry-picked on purpose; the text says so.
            # EXAMPLE: append(i).
            if src(X[i:i + 1].to(device)).argmax(1).item() == y:  # the model guesses this picture right (an attack only makes sense then)
                chosen.append(i)  # pick it
                seen.add(y)  # remember this class is done
            if len(chosen) == n:  # we have enough pictures
                break  # stop looking
    # WHAT: Fallback when no picture qualifies.
    # HOW: Take the first few pictures.
    # WHY: Never crash.
    # PROS & CONS: + Robust. - Less meaningful.
    # EXAMPLE: Only possible in tiny tests.
    if not chosen:  # nothing suitable was found
        chosen = list(range(min(n, len(X))))  # just take the first few pictures
    # WHAT: Moves the chosen pictures and labels to the device.
    # HOW: Index and copy.
    # WHY: The attack runs on the GPU.
    # PROS & CONS: + Fast. - None.
    # EXAMPLE: (4, 3, 32, 32).
    x = X[chosen].to(device)  # the chosen pictures, on the GPU/CPU
    y = Y[chosen].to(device)  # their true labels
    # WHAT: Fixes the random start.
    # HOW: Seed 42.
    # WHY: Same figure every run.
    # PROS & CONS: + Repeatable. - None.
    # EXAMPLE: The random start is identical each run, so the noise picture does not change.
    torch.manual_seed(42)  # fixed starting number so the random start is the same every time
    # WHAT: Creates the adversarial pictures.
    # HOW: PGD-10, eps = 8/255, random start.
    # WHY: Same attack as the evaluation.
    # PROS & CONS: + Consistent. - None.
    # EXAMPLE: (4, 3, 32, 32).
    x_adv = pgd_attack(src, x, y, eps, alpha=max(eps / 4, 1 / 255), steps=pgd_steps, random_start=True)  # make the tricked version of each picture
    # WHAT: Makes the noise visible.
    # HOW: (x_adv - x + eps) / (2 eps) maps [-eps, +eps] to [0, 1].
    # WHY: The real noise is only 8/255 and would look invisible.
    # PROS & CONS: + Visible. - Exaggerated (magnified).
    # EXAMPLE: 0 change maps to grey 0.5.
    pert = ((x_adv - x + eps) / (2 * eps)).clamp(0, 1)  # the real noise is too small to see, so stretch it to the range 0..1 (grey = no change)
    # WHAT: Collects every model's prediction.
    # HOW: Dictionary name to predictions.
    # WHY: Printed under each picture.
    # PROS & CONS: + Clear. - None.
    # EXAMPLE: {}
    preds = {}  # what each model guesses for the tricked pictures
    # WHAT: Predicts the adversarial pictures with all four models.
    # HOW: argmax of the 10 scores.
    # WHY: Shows who is fooled.
    # PROS & CONS: + Direct. - None.
    # EXAMPLE: CNN-Clean: dog [WRONG].
    with torch.no_grad():  # no learning information needed here
        for name, m in models.items():  # go through all the models
            preds[name] = m.eval()(x_adv).argmax(1).cpu().tolist()  # the guess of this model for each tricked picture
    # WHAT: One row of the figure per picture.
    # HOW: Count of chosen pictures.
    # WHY: Layout.
    # PROS & CONS: + Simple. - None.
    # EXAMPLE: 4
    rows = len(chosen)  # one row of the graph per picture
    # WHAT: Canvas with 3 columns.
    # HOW: Original, noise, adversarial.
    # WHY: Story: original, noise, result.
    # PROS & CONS: + Clear. - None.
    # EXAMPLE: (4, 3) axes.
    fig, axes = plt.subplots(rows, 3, figsize=(10.5, 3.1 * rows))  # a canvas with 3 columns: original, noise, tricked picture
    # WHAT: Makes axes 2-D even for one row.
    # HOW: numpy.
    # WHY: Same indexing code.
    # PROS & CONS: + Robust. - None.
    # EXAMPLE: axes[0, 0].
    axes = np.atleast_2d(axes)  # makes sure indexing works even when there is only one row
    # WHAT: Fills one row per picture.
    # HOW: Three images and titles.
    # WHY: Presentation.
    # PROS & CONS: + Clear. - None.
    # EXAMPLE: 4 rows.
    for r in range(rows):  # go through each picture
        # WHAT: Draws the original picture.
        # HOW: Channel order changed to height x width x colour.
        # WHY: matplotlib expects that order.
        # PROS & CONS: + Correct colours. - None.
        # EXAMPLE: Title 'Clean: frog'.
        axes[r, 0].imshow(x[r].permute(1, 2, 0).cpu().numpy())  # show the original picture (colour layers moved to the order the drawing tool wants)
        axes[r, 0].set_title(f"Clean: {CIFAR10_CLASSES[int(y[r])]}", fontsize=10)  # title: its true class name
        # WHAT: Draws the magnified noise.
        # HOW: Same layout.
        # WHY: Shows what the attack added.
        # PROS & CONS: + Clear. - Magnified.
        # EXAMPLE: Colourful noise.
        axes[r, 1].imshow(pert[r].permute(1, 2, 0).cpu().numpy())  # show the stretched noise
        axes[r, 1].set_title(r"PGD-10 perturbation, magnified $(x_{adv}-x+\epsilon)/2\epsilon$", fontsize=8)  # title: this is the magnified noise
        # WHAT: Draws the adversarial picture.
        # HOW: Detach from autograd first.
        # WHY: Shows it looks unchanged.
        # PROS & CONS: + Clear. - None.
        # EXAMPLE: Same frog.
        axes[r, 2].imshow(x_adv[r].permute(1, 2, 0).detach().cpu().numpy())  # show the tricked picture
        # WHAT: Prediction text for the title.
        # HOW: One line per model with OK or WRONG.
        # WHY: Readers see who is fooled.
        # PROS & CONS: + Informative. - Small font.
        # EXAMPLE: 'CNN-Clean: dog [WRONG]'.
        lines = []  # text lines with each model's guess
        for name in models:  # go through all the models
            p = preds[name][r]  # this model's guess
            ok = "OK" if p == int(y[r]) else "WRONG"  # is the guess right?
            lines.append(f"{name}: {CIFAR10_CLASSES[p]} [{ok}]")  # write: model name, guessed class, right or wrong
        axes[r, 2].set_title(f"x_adv (eps=8/255, crafted on {source})\n" + "\n".join(lines), fontsize=7)  # title: all models' guesses
        # WHAT: Hides the axes.
        # HOW: axis('off').
        # WHY: Cleaner pictures.
        # PROS & CONS: + Clean. - None.
        # EXAMPLE: axes[r, 0].axis('off') removes the tick numbers around the first picture.
        for c in range(3):  # go through the 3 columns
            axes[r, c].axis("off")  # hide the numbers around the picture
    # WHAT: Layout, optional note, save, close.
    # HOW: tight_layout fits the labels, _tag writes the optional note, savefig writes the PNG at 300 DPI, close
    #      frees the figure.
    # WHY: Clipped labels and leaked figures are the usual matplotlib problems in long runs; the note stops
    #      quick-test images being mistaken for results.
    # PROS & CONS: + Clean output. - None.
    # EXAMPLE: outputs/adversarial_examples_grid.png.
    fig.tight_layout(rect=(0, 0.02, 1, 1))  # fit everything neatly
    _tag(fig, tag)  # add the small grey note if there is one
    fig.savefig(path, dpi=DPI)  # save the graph as an image file
    plt.close(fig)  # release the memory


# WHAT: Draws all four figures and returns their paths.
# HOW: Calls the four plot functions in turn.
# WHY: One call from main.py.
# PROS & CONS: + Convenient. - None.
# EXAMPLE: dict of four paths.
def make_all_plots(sweep_df, transfer_df, models, X, Y, device, out_dir, pgd_steps=10, tag=""):  # draws all four graphs and gives back their file paths
    # WHAT: Creates the output folder.
    # HOW: exist_ok.
    # WHY: Avoid errors.
    # PROS & CONS: + Safe. - None.
    # EXAMPLE: outputs/
    os.makedirs(out_dir, exist_ok=True)  # create the output folder if it is missing
    # WHAT: File names for the four figures.
    # HOW: Dictionary.
    # WHY: Same names as in the TASK list.
    # PROS & CONS: + Predictable. - None.
    # EXAMPLE: outputs/acc_vs_epsilon.png
    paths = {  # file name for each graph
        "acc_vs_epsilon": os.path.join(out_dir, "acc_vs_epsilon.png"),  # graph 1
        "tradeoff": os.path.join(out_dir, "clean_vs_robust_tradeoff.png"),  # graph 2
        "transfer": os.path.join(out_dir, "transferability_heatmap.png"),  # graph 3
        "examples": os.path.join(out_dir, "adversarial_examples_grid.png"),  # graph 4
    }
    # WHAT: Draws the four figures.
    # HOW: One function call per figure.
    # WHY: Each saves its own PNG.
    # PROS & CONS: + Clear. - None.
    # EXAMPLE: 4 PNG files.
    plot_acc_vs_epsilon(sweep_df, paths["acc_vs_epsilon"], tag)  # draw graph 1
    plot_tradeoff(sweep_df, paths["tradeoff"], tag)  # draw graph 2
    plot_transferability(transfer_df, paths["transfer"], tag)  # draw graph 3
    plot_adversarial_examples(models, X, Y, device, paths["examples"], pgd_steps=pgd_steps, tag=tag)  # draw graph 4
    # WHAT: Tells the user where the files are.
    # HOW: Print each path.
    # WHY: Convenient.
    # PROS & CONS: + Visible. - None.
    # EXAMPLE: [plots] saved outputs/acc_vs_epsilon.png
    for p in paths.values():  # go through the saved files
        print(f"[plots] saved {p}")  # tell the user where each one was saved
    # WHAT: Returns the paths.
    # HOW: Dictionary.
    # WHY: Callers can open them.
    # PROS & CONS: + Convenient. - None.
    # EXAMPLE: paths['transfer']
    return paths  # give back the file paths

In [ ]:
%%writefile main.py
"""CLI entry point: CNN vs ViT adversarial robustness on CIFAR-10.

  python main.py --quick-test            # CPU smoke test (64 test images, 1 tiny epoch), no GPU needed
  python main.py --full-run              # Colab/Kaggle GPU: 10-epoch adversarial fine-tuning + full evaluation
  python main.py --epochs 10 --eval-samples 1000 --auto-push

Checkpoints in checkpoints/ are reused automatically: nothing that already exists is retrained.
"""
# WHAT: Imports the standard libraries used by the command-line program.
# HOW: argparse reads options, copy makes independent model copies, json prints results, os handles paths, time
#      measures the run.
# WHY: main.py coordinates training and evaluation and needs these basic tools.
# PROS & CONS: + Standard library only. - None.
# EXAMPLE: copy.deepcopy(model) clones a model.
import argparse  # read options typed in the terminal
import copy  # make independent copies of models
import json  # print / save results as JSON text
import os  # work with file and folder paths
import time  # measure how long the run takes

# WHAT: Imports the project's own building blocks.
# HOW: Data loading, training, evaluation, model loading, utilities and plotting come from src/.
# WHY: main.py contains no algorithms; it only orders the steps.
# PROS & CONS: + Clear separation. - Import errors appear at start-up if a file is missing.
# EXAMPLE: from src.attacks is imported indirectly by defense.py and evaluate.py.
from src.data import get_eval_tensors, get_train_loader, prepare_data  # functions that get the pictures
from src.defense import adversarial_finetune, clean_finetune, is_training_complete  # functions that train the models
from src.evaluate import accuracy, evaluate_epsilon_sweep, evaluate_transferability, export_results  # functions that test the models and save results
from src.models import load_checkpoint, load_cnn_clean, load_vit_clean, save_checkpoint  # functions that create, load and save models
from src.utils import (SEED, get_device, git_auto_push, load_json, restore_from_drive, save_json, set_seed,  # small helper tools
                       sync_to_drive)  # backup to Google Drive
from src.visualize import make_all_plots  # function that draws the graphs


# WHAT: Defines the command-line options.
# HOW: argparse turns text such as '--epochs 5' into attributes of an args object.
# WHY: The same file serves a 2-minute smoke test and the full GPU run.
# PROS & CONS: + Flexible, self-documenting (--help). - Many options to remember.
# EXAMPLE: python main.py --full-run --epochs 10 --eval-samples 1000
def parse_args():  # reads the options typed after "python main.py", e.g. --epochs 5
    # WHAT: Creates the option reader.
    # HOW: RawDescriptionHelpFormatter prints the module docstring unchanged for --help.
    # WHY: Keeps the usage examples readable.
    # PROS & CONS: + Good --help text. - None.
    # EXAMPLE: python main.py --help
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)  # the option reader; shows the text at the top of this file when asked for help
    # WHAT: Groups the two run modes so only one can be chosen.
    # HOW: argparse rejects '--quick-test --full-run'.
    # WHY: A run is either a smoke test or a real run.
    # PROS & CONS: + Prevents contradictory options. - None.
    # EXAMPLE: python main.py --quick-test
    mode = p.add_mutually_exclusive_group()  # options in this group cannot be used together
    # WHAT: Option for the tiny CPU smoke test.
    # HOW: Flag without value; True when given.
    # WHY: Checks the whole pipeline in minutes before a long GPU run.
    # PROS & CONS: + Cheap safety net. - Quick-test numbers are NOT results (marked on the plots).
    # EXAMPLE: 64 test images, 1 epoch, 2 PGD steps.
    mode.add_argument("--quick-test", action="store_true", help="CPU smoke test: 64 images, 1 short epoch")  # tiny fast run just to check the code works
    # WHAT: Option for the real run (the default).
    # HOW: Flag without value.
    # WHY: Makes the intention explicit in notebooks and logs.
    # PROS & CONS: + Clear. - None.
    # EXAMPLE: python main.py --full-run
    mode.add_argument("--full-run", action="store_true", help="Full GPU run (default when no mode is given)")  # the real run
    # WHAT: Number of adversarial fine-tuning epochs.
    # HOW: Integer, default 10.
    # WHY: 10 epochs from pretrained weights are enough to see the robustness gain and fit a free GPU.
    # PROS & CONS: + Cheap. - Shorter than training robust models from scratch.
    # EXAMPLE: --epochs 10
    p.add_argument("--epochs", type=int, default=10, help="adversarial fine-tuning epochs")  # how many times the model sees the whole training set
    # WHAT: Size of the stratified test subset.
    # HOW: Integer, default 1000 (100 per class).
    # WHY: Attacks are expensive; 1000 images give a standard error of up to 1.6 points.
    # PROS & CONS: + Fast evaluation. - Not the full 10,000 test images.
    # EXAMPLE: --eval-samples 1000
    p.add_argument("--eval-samples", type=int, default=1000, help="stratified test subset size")  # how many test images are used for testing
    # WHAT: Training epsilon in /255 units.
    # HOW: Float, default 8.
    # WHY: 8/255 is the common CIFAR-10 benchmark budget.
    # PROS & CONS: + Standard. - Robustness at other budgets is weaker (e.g. 16/255).
    # EXAMPLE: --eps 8 means 8/255 = 0.0314.
    p.add_argument("--eps", type=float, default=8.0, help="training epsilon in /255 units")  # biggest change per pixel used while training the defence
    # WHAT: Which attack trains the CNN defence.
    # HOW: Choice of pgd5 or rs_fgsm.
    # WHY: PGD-5 is the stronger, safer choice.
    # PROS & CONS: + Switchable. - rs_fgsm risks catastrophic overfitting.
    # EXAMPLE: default pgd5.
    p.add_argument("--cnn-method", choices=["pgd5", "rs_fgsm"], default="pgd5")  # which attack trains the CNN's defence
    # WHAT: Which attack trains the ViT defence.
    # HOW: Same choices; default pgd5 so the CNN and the ViT are treated alike.
    # WHY: The first ViT run with rs_fgsm showed catastrophic overfitting (79.8% FGSM, 4.9% PGD-10).
    # PROS & CONS: + Fair comparison. - PGD-5 is about 2.5x slower for the ViT (74.7 vs 30.1 minutes).
    # EXAMPLE: default pgd5.
    p.add_argument("--vit-method", choices=["pgd5", "rs_fgsm"], default="pgd5",  # which attack trains the ViT's defence
                   help="pgd5 matches the CNN (fair comparison). rs_fgsm is ~2.5x faster but the ViT showed "  # help text, part 1
                        "signs of catastrophic overfitting with it (see outputs_vit_rsfgsm/)")  # help text, part 2
    # WHAT: Which ViT to use.
    # HOW: timm-tiny (5.7M weights) or hf-base (much larger).
    # WHY: Tiny is the only option that fits a free Colab session.
    # PROS & CONS: + Choice. - hf-base is slow and its weights exceed GitHub's file limit.
    # EXAMPLE: default timm-tiny.
    p.add_argument("--vit-backbone", choices=["timm-tiny", "hf-base"], default="timm-tiny")  # which ViT to use: small (fast) or big (slow)
    # WHAT: Epochs of normal fine-tuning for the new ViT head.
    # HOW: Integer, default 5.
    # WHY: A new 10-class head must learn CIFAR-10 before adversarial training.
    # PROS & CONS: + Reaches 97.1% clean accuracy. - Extra time.
    # EXAMPLE: --vit-clean-epochs 5
    p.add_argument("--vit-clean-epochs", type=int, default=5, help="clean fine-tune epochs for timm ViT-Tiny")  # epochs of normal training for the small ViT
    # WHAT: Learning rate.
    # HOW: Float, default 1e-4.
    # WHY: Gentle updates for fine-tuning pretrained weights.
    # PROS & CONS: + Stable. - May be too small for training from scratch.
    # EXAMPLE: --lr 0.0001
    p.add_argument("--lr", type=float, default=1e-4)  # learning rate = how big each weight update is
    # WHAT: Images per training step.
    # HOW: Integer, default 128.
    # WHY: Fits the T4 even with the doubled (clean + adversarial) batch.
    # PROS & CONS: + Good speed/memory balance. - Larger batches need a larger learning rate.
    # EXAMPLE: 128 images, 256 including adversarial copies.
    p.add_argument("--batch-size", type=int, default=128)  # how many images are processed together
    # WHAT: Optional limit on training images.
    # HOW: 0 means use all 50,000.
    # WHY: Used only for tests.
    # PROS & CONS: + Fast tests. - Must stay 0 for results.
    # EXAMPLE: --max-train-samples 64
    p.add_argument("--max-train-samples", type=int, default=0, help="0 = full 50k training set")  # use only part of the training images (0 = use all)
    # WHAT: Helper processes for loading images.
    # HOW: Integer, default 2.
    # WHY: Colab has 2 CPU cores.
    # PROS & CONS: + Faster loading. - More processes use more RAM.
    # EXAMPLE: --num-workers 2
    p.add_argument("--num-workers", type=int, default=2)  # how many helper processes load images
    # WHAT: Folder, results folder and saved-model folder options.
    # HOW: Plain paths with defaults data, outputs and checkpoints.
    # WHY: Lets notebooks and tests use different folders.
    # PROS & CONS: + Flexible. - Mistakes can mix quick-test and real files.
    # EXAMPLE: --out-dir outputs_vit_rsfgsm
    p.add_argument("--data-root", default="data")  # folder for the dataset
    p.add_argument("--ckpt-dir", default=None)  # folder for saved models
    p.add_argument("--out-dir", default="outputs")  # folder for result tables and graphs
    # WHAT: Google Drive backup folder.
    # HOW: Optional path under /content/drive.
    # WHY: Colab disks are wiped at the end of a session.
    # PROS & CONS: + Lose nothing on a disconnect. - Needs Drive mounted.
    # EXAMPLE: --drive-dir /content/drive/MyDrive/adv_ckpts
    p.add_argument("--drive-dir", default=None, help="e.g. /content/drive/MyDrive/adv_ckpts (Colab)")  # Google Drive folder for backups
    # WHAT: Ignore existing adversarial checkpoints.
    # HOW: Boolean flag.
    # WHY: Needed when settings change.
    # PROS & CONS: + Explicit. - Deletes progress.
    # EXAMPLE: --force-retrain
    p.add_argument("--force-retrain", action="store_true", help="ignore existing adversarial checkpoints")  # train again even if saved models exist
    # WHAT: Upload progress to GitHub automatically.
    # HOW: Boolean flag; needs a token.
    # WHY: Backup of long runs.
    # PROS & CONS: + Remote copy. - Token handling must stay secret.
    # EXAMPLE: --auto-push
    p.add_argument("--auto-push", action="store_true", help="git commit+push after each stage / every 5 epochs")  # upload progress to GitHub automatically
    # WHAT: Reads and returns the typed options.
    # HOW: argparse parses sys.argv.
    # WHY: main() uses them as args.<name>.
    # PROS & CONS: + Standard. - None.
    # EXAMPLE: args.epochs == 10.
    return p.parse_args()  # read the typed options and give them back


# WHAT: Deletes old quick-test result files.
# HOW: Reads outputs/run_info.json; if it says mode == quick-test, removes the generated files.
# WHY: Quick-test numbers (64 images, 2 PGD steps) must never be mistaken for results.
# PROS & CONS: + Prevents wrong numbers in the report. - Deletes files, so only quick-test files are removed.
# EXAMPLE: '[outputs] removed stale quick-test artifacts'.
def clear_stale_quick_test_outputs(out_dir):  # deletes old quick-test results so fake numbers never mix with real ones
    """Quick-test artifacts are committed to git; delete them so they can never pass for full-run results."""
    # WHAT: Reads which kind of run produced the files.
    # HOW: Loads run_info.json (empty dict if missing).
    # WHY: Decides whether the files are quick-test output.
    # PROS & CONS: + Safe. - Relies on run_info.json being written.
    # EXAMPLE: {'mode': 'quick-test', ...}
    info = load_json(os.path.join(out_dir, "run_info.json"), {}) or {}  # read the note that says which kind of run made the files
    # WHAT: Stops unless the files are from a quick test.
    # HOW: Compares the stored mode.
    # WHY: Real results are never deleted.
    # PROS & CONS: + Safe. - None.
    # EXAMPLE: mode == 'full-run' returns early.
    if info.get("mode") != "quick-test":  # the files are from a real run (or missing)
        return  # keep them
    # WHAT: Deletes the known generated files.
    # HOW: Loop over names, os.remove if present.
    # WHY: Clean slate before a real run.
    # PROS & CONS: + Precise list. - Must be updated when new outputs are added.
    # EXAMPLE: outputs/results_summary.csv removed.
    for name in ("acc_vs_epsilon.png", "clean_vs_robust_tradeoff.png", "transferability_heatmap.png",  # the files made by a quick test
                 "adversarial_examples_grid.png", "results_summary.csv", "results_summary.md",  # more of those files
                 "transferability.csv", "run_info.json"):  # and the last ones
        p = os.path.join(out_dir, name)  # full path of the file
        if os.path.exists(p):  # the file exists
            os.remove(p)  # delete it
    # WHAT: Tells the user what happened.
    # HOW: Console message.
    # WHY: Transparency.
    # PROS & CONS: + Visible. - None.
    # EXAMPLE: [outputs] removed stale quick-test artifacts
    print("[outputs] removed stale quick-test artifacts")  # tell the user


# WHAT: The whole experiment in order.
# HOW: Load models, train what is missing, evaluate, save tables, draw figures, write a run summary.
# WHY: A single entry point makes the experiment repeatable.
# PROS & CONS: + Resumable; nothing existing is retrained. - Long function.
# EXAMPLE: python main.py --full-run took 111 minutes in total for the final ViT run.
def main():  # the main program: runs all the steps in order
    # WHAT: Reads the options and sets the mode.
    # HOW: quick is True for the smoke test.
    # WHY: Used throughout main().
    # PROS & CONS: + Simple. - None.
    # EXAMPLE: quick = False for the real run.
    args = parse_args()  # read the typed options
    quick = args.quick_test  # True if this is the tiny test run
    # WHAT: Chooses GPU or CPU and fixes the random seed.
    # HOW: get_device and set_seed(42).
    # WHY: Speed and repeatability.
    # PROS & CONS: + Portable. - CPU is far too slow for a full run.
    # EXAMPLE: cuda on Colab.
    device = get_device()  # GPU if there is one, else CPU
    set_seed(SEED)  # fix the random numbers so results can be repeated
    # WHAT: Default settings of a real run.
    # HOW: PGD uses 10 steps, no batch limit, no note on the figures.
    # WHY: Quick tests override them below.
    # PROS & CONS: + Clear defaults. - None.
    # EXAMPLE: pgd_steps = 10.
    pgd_steps, max_batches, tag = 10, 0, ""  # normal settings: PGD uses 10 steps, no batch limit, no note on graphs
    # WHAT: Shrinks everything for the smoke test.
    # HOW: 1 epoch, 64 test images, batch 16, 2 PGD steps, 2 batches per epoch.
    # WHY: Runs in a few minutes on a CPU.
    # PROS & CONS: + Fast verification. - Numbers are invalid as results.
    # EXAMPLE: Plots get the note 'QUICK-TEST ... not a result'.
    if quick:  # tiny test run: make everything very small
        args.epochs, args.eval_samples, args.batch_size = 1, 64, 16  # 1 epoch, 64 test images, 16 images per batch
        args.max_train_samples, args.vit_clean_epochs, args.num_workers = 64, 1, 0  # 64 training images, 1 normal-training epoch, no helper processes
        args.drive_dir = None  # do not touch the real Google Drive backups
        pgd_steps, max_batches = 2, 2  # only 2 attack steps and 2 batches per epoch
        tag = "QUICK-TEST smoke run (tiny N, 1 epoch, 2 PGD steps) - not a result"  # note printed on the graphs
    # WHAT: Chooses and creates the folders.
    # HOW: Quick tests use checkpoints_quick_test so real checkpoints are never mixed with fake ones.
    # WHY: Protects real results.
    # PROS & CONS: + Safe. - Two folders to know.
    # EXAMPLE: checkpoints/ for real runs.
    args.ckpt_dir = args.ckpt_dir or ("checkpoints_quick_test" if quick else "checkpoints")  # saved models go to a separate folder for quick tests
    # WHAT: Creates the folders if missing.
    # HOW: makedirs with exist_ok.
    # WHY: Avoids errors.
    # PROS & CONS: + Safe. - None.
    # EXAMPLE: outputs/ appears on the first run.
    os.makedirs(args.ckpt_dir, exist_ok=True)  # create the saved-models folder
    os.makedirs(args.out_dir, exist_ok=True)  # create the results folder
    # WHAT: Converts eps to pixel units.
    # HOW: 8 becomes 8/255 = 0.0314.
    # WHY: Images are in [0, 1].
    # PROS & CONS: + Correct scale. - None.
    # EXAMPLE: 0.03137
    eps_train = args.eps / 255.0  # change e.g. 8 into 8/255, because pixel values are between 0 and 1

    # WHAT: Prints the setup.
    # HOW: Device, mode, checkpoint folder.
    # WHY: Shows what is about to run.
    # PROS & CONS: + Transparent. - None.
    # EXAMPLE: Device: cuda | mode: full-run | ckpt: checkpoints
    print(f"Device: {device} | mode: {'quick-test' if quick else 'full-run'} | ckpt: {args.ckpt_dir}")  # show the setup
    # WHAT: Warns about a full run on a CPU.
    # HOW: Console warning.
    # WHY: Would take days.
    # PROS & CONS: + Prevents wasted time. - Does not stop the run.
    # EXAMPLE: WARNING: full-run on CPU will be extremely slow.
    if device.type == "cpu" and not quick:  # a full run without a GPU
        print("WARNING: full-run on CPU will be extremely slow. Use Colab/Kaggle GPU (or --quick-test).")  # warn the user

    # WHAT: Restores saved models from Drive and clears old quick-test outputs.
    # HOW: Copies missing files back; deletes stale quick-test files in real runs.
    # WHY: A new Colab session can continue.
    # PROS & CONS: + No retraining. - Needs Drive.
    # EXAMPLE: cnn_adv.pth restored.
    restore_from_drive(args.ckpt_dir, args.drive_dir)  # bring back saved models from Google Drive (if used)
    # WHAT: Cleanup only for real runs.
    # HOW: Calls clear_stale_quick_test_outputs.
    # WHY: Real runs start from clean outputs.
    # PROS & CONS: + Safe. - None.
    # EXAMPLE: In a real run an old quick-test run_info.json triggers the deletion of the fake result files.
    if not quick:  # real run
        clear_stale_quick_test_outputs(args.out_dir)  # remove old quick-test results
    # WHAT: Defines the upload function.
    # HOW: A function that uploads, or does nothing without --auto-push.
    # WHY: Call sites stay the same either way.
    # PROS & CONS: + Simple. - None.
    # EXAMPLE: push('results: evaluation tables and plots').
    push = (lambda msg: git_auto_push(msg)) if args.auto_push else (lambda msg: None)  # a function that uploads to GitHub, or does nothing if auto-push is off
    # WHAT: Starts the stopwatch.
    # HOW: Wall-clock time.
    # WHY: total_minutes is recorded in run_info.json.
    # PROS & CONS: + Cheap. - Includes download time (22 minutes once).
    # EXAMPLE: 111.05 minutes for the final run.
    t_start = time.time()  # note the start time

    # WHAT: Paths of the clean model files.
    # HOW: cnn_clean.pth and vit_clean.pth in the checkpoint folder.
    # WHY: Saved clean models make runs repeatable.
    # PROS & CONS: + Reuse. - None.
    # EXAMPLE: checkpoints/cnn_clean.pth
    cnn_path = os.path.join(args.ckpt_dir, "cnn_clean.pth")  # file of the normal CNN's weights
    vit_path = os.path.join(args.ckpt_dir, "vit_clean.pth")  # file of the normal ViT's weights

    # WHAT: Loads or creates the clean CNN.
    # HOW: If saved: build an empty ResNet-20 and fill it with the saved weights; otherwise download the
    #      pretrained CIFAR-10 weights and save them.
    # WHY: The same weights in every session.
    # PROS & CONS: + Repeatable. - The CNN was never trained from scratch by us.
    # EXAMPLE: '[CNN-Clean] loaded from checkpoint'.
    if os.path.exists(cnn_path):  # the CNN was already saved earlier
        cnn_clean = load_checkpoint(load_cnn_clean(device, pretrained=False), cnn_path, device)  # build an empty CNN and fill it with the saved weights
        print("[CNN-Clean] loaded from checkpoint")  # tell the user
    else:  # first run
        cnn_clean = load_cnn_clean(device)  # download the ready-made CNN
        save_checkpoint(cnn_clean, cnn_path)  # save it for next time

    # WHAT: Loads or creates the clean ViT.
    # HOW: If saved: load. Otherwise download ImageNet weights and remember it needs clean fine-tuning.
    # WHY: A new classifier head has random weights.
    # PROS & CONS: + Skips repeated fine-tuning. - A lost checkpoint means a second fine-tune (97.4% vs 97.1%).
    # EXAMPLE: '[ViT-Clean] loaded from checkpoint'.
    vit_needs_ft = False  # does the ViT still need normal training? assume no
    # WHAT: Loads the saved clean ViT when available.
    # HOW: Builds an untrained shell and loads the saved weights.
    # WHY: Avoids repeating the 5-epoch fine-tune.
    # PROS & CONS: + Saves time. - Needs the same architecture.
    # EXAMPLE: vit_clean.pth.
    if os.path.exists(vit_path):  # the ViT was already saved earlier
        shell, _ = load_vit_clean(device, args.vit_backbone, pretrained=False)  # build an empty ViT
        vit_clean = load_checkpoint(shell, vit_path, device)  # fill it with the saved weights
        print("[ViT-Clean] loaded from checkpoint")  # tell the user
    else:  # first run
        vit_clean, vit_needs_ft = load_vit_clean(device, args.vit_backbone)  # download the ViT and learn if it needs normal training

    # WHAT: Downloads CIFAR-10 if needed.
    # HOW: Calls prepare_data.
    # WHY: Training needs the data.
    # PROS & CONS: + Cached. - First download is slow.
    # EXAMPLE: data/cifar-10-batches-py.
    prepare_data(args.data_root, args.drive_dir)  # make sure the dataset is downloaded
    # WHAT: Builds the training loader.
    # HOW: Shuffled batches of 128 with augmentation.
    # WHY: Used by both clean and adversarial training.
    # PROS & CONS: + Standard. - None.
    # EXAMPLE: 390 batches per epoch.
    train_loader = get_train_loader(args.batch_size, args.data_root, args.max_train_samples,  # the object that hands out training images in groups
                                    num_workers=args.num_workers)  # number of helper processes

    # WHAT: Clean fine-tuning of the ViT when needed.
    # HOW: 5 epochs of normal training, then save.
    # WHY: The ViT must classify CIFAR-10 before it is attacked.
    # PROS & CONS: + 97.1% clean accuracy. - Extra time.
    # EXAMPLE: vit_clean.pth saved.
    if vit_needs_ft:  # the new ViT must first learn CIFAR-10 with normal training
        print(f"[ViT-Clean] clean fine-tune on CIFAR-10 for {args.vit_clean_epochs} epochs")  # tell the user
        clean_finetune(vit_clean, train_loader, epochs=args.vit_clean_epochs, device=device,  # normal training, no tricked images
                       max_batches=max_batches)  # batch limit (only used by the quick test)
        save_checkpoint(vit_clean, vit_path)  # save the trained ViT
    # WHAT: Backs up the clean models.
    # HOW: Drive copy and optional GitHub push.
    # WHY: Safety.
    # PROS & CONS: + Safe. - None.
    # EXAMPLE: cnn_clean.pth and vit_clean.pth are copied to the Drive folder after they are created.
    sync_to_drive(args.ckpt_dir, args.drive_dir)  # back up the saved models to Google Drive (if used)
    push("chore(train): clean CNN/ViT baseline checkpoints")  # upload to GitHub (if switched on)

    # WHAT: Path of the training log.
    # HOW: JSON file in the checkpoint folder.
    # WHY: Training history for the report.
    # PROS & CONS: + Persistent. - None.
    # EXAMPLE: checkpoints/training_history.json
    hist_path = os.path.join(args.ckpt_dir, "training_history.json")  # the log file of training progress
    # WHAT: Will hold the two adversarially trained models.
    # HOW: Empty dictionary filled in the loop.
    # WHY: Evaluation needs all four models.
    # PROS & CONS: + Simple. - None.
    # EXAMPLE: {}
    adv_models = {}  # will hold the two defended models
    # WHAT: Loops over the CNN and the ViT.
    # HOW: (file tag, clean model, training attack, display name).
    # WHY: Both models get exactly the same recipe.
    # PROS & CONS: + Fair. - None.
    # EXAMPLE: ('vit_adv', vit_clean, 'pgd5', 'ViT-AdvTrained').
    for tag_name, clean, method, label in (("cnn_adv", cnn_clean, args.cnn_method, "CNN-AdvTrained"),  # first: (file name, normal model, attack to train with, display name) for the CNN
                                           ("vit_adv", vit_clean, args.vit_method, "ViT-AdvTrained")):  # second: the same four things for the ViT
        # WHAT: File of the defended model.
        # HOW: Path of the .pth file.
        # WHY: Used for skip/resume.
        # PROS & CONS: + Simple. - None.
        # EXAMPLE: checkpoints/vit_adv.pth
        ckpt = os.path.join(args.ckpt_dir, f"{tag_name}.pth")  # file of the defended model's weights
        # WHAT: Copy of the clean model to train.
        # HOW: deepcopy duplicates all weights.
        # WHY: The clean model must stay unchanged so that before/after can be compared.
        # PROS & CONS: + Clean baseline preserved. - Doubles model memory briefly.
        # EXAMPLE: ViT-Clean stays at 97.1%.
        model = copy.deepcopy(clean)  # make a full separate copy, so the normal model stays unchanged
        # WHAT: Skips training if a finished checkpoint exists.
        # HOW: Loads the weights.
        # WHY: Never repeats work.
        # PROS & CONS: + Saves up to 75 minutes. - Trusts the log file.
        # EXAMPLE: '[ViT-AdvTrained] completed checkpoint found -> skipping training'.
        if (os.path.exists(ckpt) and not args.force_retrain  # a saved defended model exists and we were not told to retrain
                and is_training_complete(hist_path, tag_name, args.epochs)):  # and the log says its training finished
            load_checkpoint(model, ckpt, device)  # just load it
            print(f"[{label}] completed checkpoint found -> skipping training")  # tell the user
        # WHAT: Trains the defended model.
        # HOW: Prints the settings, optionally deletes old files, runs adversarial_finetune.
        # WHY: Defence training.
        # PROS & CONS: + Resumable. - Long run.
        # EXAMPLE: '[ViT-AdvTrained] adversarial fine-tuning: 10 epochs, pgd5, eps=8.0/255'.
        else:  # we need to train it
            print(f"[{label}] adversarial fine-tuning: {args.epochs} epochs, {method}, eps={args.eps}/255")  # tell the user the settings
            # WHAT: Deletes old files when retraining is forced.
            # HOW: os.remove of the weights and resume state.
            # WHY: A clean restart.
            # PROS & CONS: + Explicit. - Irreversible.
            # EXAMPLE: --force-retrain.
            if args.force_retrain:  # start again from zero
                for f in (ckpt, os.path.join(args.ckpt_dir, f"{tag_name}_state.pt")):  # the saved model and the "continue later" file
                    if os.path.exists(f):  # the file exists
                        os.remove(f)  # delete it
            # WHAT: The actual defence training.
            # HOW: 10 epochs, PGD-5, mixed precision, checkpointing every epoch.
            # WHY: Main experiment.
            # PROS & CONS: + Resumable. - Time-consuming.
            # EXAMPLE: CNN 6.3 min, ViT 74.7 min on a T4.
            adversarial_finetune(model, train_loader, epochs=args.epochs, eps=eps_train, method=method,  # train the copy using normal AND tricked images
                                 lr=args.lr, device=device, tag=tag_name, ckpt_dir=args.ckpt_dir,  # learning rate, GPU/CPU, name, folder for saved files
                                 drive_dir=args.drive_dir, auto_push_every=5 if args.auto_push else 0,  # Drive backup folder; upload to GitHub every 5 epochs if switched on
                                 max_batches=max_batches)  # batch limit (only used by the quick test)
            # WHAT: Uploads progress after training.
            # HOW: GitHub push (if enabled).
            # WHY: Remote backup.
            # PROS & CONS: + Safe. - None.
            # EXAMPLE: Commit message: 'chore(train): ViT-AdvTrained 10-epoch adversarial fine-tuning finished'.
            push(f"chore(train): {label} {args.epochs}-epoch adversarial fine-tuning finished")  # upload to GitHub (if switched on)
        # WHAT: Stores the defended model in test mode.
        # HOW: eval().
        # WHY: Ready for evaluation.
        # PROS & CONS: + Clear. - None.
        # EXAMPLE: adv_models['CNN-AdvTrained'].
        adv_models[label] = model.eval()  # keep this defended model, in test mode

    # WHAT: All four models together.
    # HOW: Two clean, two defended.
    # WHY: The core comparison.
    # PROS & CONS: + Clear. - None.
    # EXAMPLE: dict with 4 entries.
    models = {"CNN-Clean": cnn_clean, "ViT-Clean": vit_clean, **adv_models}  # all four models together: 2 normal + 2 defended
    # WHAT: Freezes every weight.
    # HOW: requires_grad_(False).
    # WHY: Attacks need gradients only for the image; saves memory and time.
    # PROS & CONS: + Faster attacks. - Models can no longer be trained.
    # EXAMPLE: No weights change during evaluation.
    for m in models.values():  # go through the four models
        for prm in m.parameters():  # go through every learnable number in the model
            prm.requires_grad_(False)  # freeze them: attacks only need to change the image, not the model (saves memory and time)

    # WHAT: Builds the fixed evaluation subset.
    # HOW: 1000 stratified test images, seed 42.
    # WHY: Every model sees the same images.
    # PROS & CONS: + Fair. - Subset only.
    # EXAMPLE: (1000, 3, 32, 32).
    X, Y = get_eval_tensors(args.eval_samples, args.data_root)  # the fixed test images X and their true labels Y
    print(f"Evaluation subset: {len(X)} stratified test images (seed {SEED})")  # tell the user how many test images are used
    # WHAT: Accuracy versus epsilon.
    # HOW: FGSM and PGD-10 at eps 0, 1, 2, 4, 8, 16 for all four models.
    # WHY: The mentor's required curves.
    # PROS & CONS: + Complete. - Slow for the ViT.
    # EXAMPLE: 48 rows.
    sweep = evaluate_epsilon_sweep(models, X, Y, device, pgd_steps=pgd_steps)  # test every model against every attack at every strength
    # WHAT: Transfer matrix at eps = 8/255.
    # HOW: PGD-10 crafted on each model, tested on each.
    # WHY: Black-box view.
    # PROS & CONS: + Extra evidence. - Only one eps.
    # EXAMPLE: 4x4 table.
    transfer = evaluate_transferability(models, X, Y, device, eps=8 / 255, pgd_steps=pgd_steps)  # test whether tricks made on one model fool the others
    # WHAT: Note text for the Markdown report.
    # HOW: Marks quick-test output as invalid.
    # WHY: Honesty about what the numbers are.
    # PROS & CONS: + Clear. - None.
    # EXAMPLE: N=1000 stratified CIFAR-10 test images, seed 42, PGD-10.
    note = (f"Quick-test smoke run only (N={len(X)}, {pgd_steps} PGD steps): not valid results."  # note text for a quick test
            if quick else f"N={len(X)} stratified CIFAR-10 test images, seed {SEED}, PGD-{pgd_steps}.")  # note text for a real run
    # WHAT: Writes the tables to files.
    # HOW: CSV and Markdown.
    # WHY: Tracked in git.
    # PROS & CONS: + Visible on GitHub. - None.
    # EXAMPLE: outputs/results_summary.csv
    export_results(sweep, transfer, args.out_dir, note)  # save the result tables as files

    # WHAT: Draws the four figures.
    # HOW: make_all_plots.
    # WHY: Visual results.
    # PROS & CONS: + Report-ready. - None.
    # EXAMPLE: outputs/*.png.
    make_all_plots(sweep, transfer, models, X, Y, device, args.out_dir, pgd_steps=pgd_steps, tag=tag)  # draw and save the four graphs

    # WHAT: Summary of the run.
    # HOW: Mode, device, epochs, settings, clean accuracy of each model, total minutes.
    # WHY: Documents provenance of the results.
    # PROS & CONS: + Traceable. - None.
    # EXAMPLE: outputs/run_info.json.
    info = {"mode": "quick-test" if quick else "full-run", "device": str(device), "epochs": args.epochs,  # a summary of this run: kind of run, device, epochs
            "eval_samples": len(X), "pgd_eval_steps": pgd_steps, "cnn_method": args.cnn_method,  # number of test images, attack steps, CNN training attack
            "vit_method": args.vit_method, "vit_backbone": args.vit_backbone,  # ViT training attack, which ViT
            "clean_acc_full_subset": {k: accuracy(m, X, Y, device) for k, m in models.items()},  # normal accuracy of each model
            "total_minutes": round((time.time() - t_start) / 60, 2)}  # total time in minutes
    # WHAT: Saves the summary.
    # HOW: JSON.
    # WHY: Used by the clean-up check and the notebook.
    # PROS & CONS: + Persistent. - None.
    # EXAMPLE: run_info.json.
    save_json(info, os.path.join(args.out_dir, "run_info.json"))  # save the summary to a file
    # WHAT: Shows the summary.
    # HOW: indent=2.
    # WHY: Visible in Colab.
    # PROS & CONS: + Visible. - None.
    # EXAMPLE: Prints the run_info dictionary, e.g. {'mode': 'full-run', 'epochs': 10, ...}.
    print(json.dumps(info, indent=2))  # show the summary on screen
    # WHAT: Uploads results.
    # HOW: GitHub push (if enabled).
    # WHY: Final backup.
    # PROS & CONS: + Safe. - None.
    # EXAMPLE: Commits outputs/*.csv, *.md and *.png with the message 'results: evaluation tables and plots'.
    push("results: evaluation tables and plots")  # upload results to GitHub (if switched on)
    # WHAT: Final message.
    # HOW: Run time.
    # WHY: Closure.
    # PROS & CONS: + Clear. - None.
    # EXAMPLE: Done in 111.05 min. Outputs in outputs/
    print(f"Done in {info['total_minutes']} min. Outputs in {args.out_dir}/")  # tell the user it is finished


# WHAT: Starts main() when the file is run.
# HOW: Python sets __name__ to '__main__'.
# WHY: Importable without side effects.
# PROS & CONS: + Standard. - None.
# EXAMPLE: python main.py
if __name__ == "__main__":  # true only when this file is run directly (python main.py)
    main()  # start the program

## Sanity checks

In [ ]:
from src.models import sanity_check  # the quick model check
from src.attacks import self_test  # the quick attack check
sanity_check()  # check both models give 10 scores per image
self_test()  # check the attacks follow their rules

## Optional quick smoke test

In [ ]:
if RUN_QUICK_TEST:  # only if the tiny test was asked for
    # next line runs the tiny test
    !python main.py --quick-test

## Full run (train + evaluate + plots)

In [ ]:
extra = ""  # extra options to add to the command below
if DRIVE_DIR: extra += f" --drive-dir '{DRIVE_DIR}'"  # add the Google Drive backup folder if we have one
if AUTO_PUSH: extra += " --auto-push"  # add automatic GitHub upload if switched on
# next line runs the whole project: loads saved models if they exist, otherwise trains, then tests and draws graphs
!python main.py --full-run --epochs 10 --eval-samples 1000 {extra}

## Results

In [ ]:
import json  # read JSON text files
from IPython.display import Image, Markdown, display  # tools to show pictures and text in the notebook
info = json.load(open("outputs/run_info.json"))  # read the summary of the last run
assert info["mode"] == "full-run", "outputs/ holds a quick-test run, not full-run results. Re-run the training cell."  # stop with an error if the results are only from the tiny test
for f in ["acc_vs_epsilon", "clean_vs_robust_tradeoff", "transferability_heatmap", "adversarial_examples_grid"]:  # the four graphs
    display(Image(filename=f"outputs/{f}.png"))  # show this graph
display(Markdown(open("outputs/results_summary.md", encoding="utf-8").read()))  # show the results tables

In [ ]:
from src.utils import git_auto_push  # the function that uploads to GitHub
if AUTO_PUSH:  # only if uploading is switched on
    git_auto_push("results(final): push complete CIFAR-10 CNN vs ViT robustness metrics and 300-DPI plots [Milestone 7]")  # upload the final results